In [ ]:
#@title 🪄 TextSwap Studio (Run this cell) { display-mode: "form" }
"""
textswap - find text inside a document image (or scanned PDF) with OCR, then
REMOVE or REPLACE it while keeping everything else untouched.

What it preserves
  * every pixel outside the edited word(s) stays bit-identical
  * background colour / gradient / paper grain is rebuilt under the old text
  * lines, table borders, underlines and neighbouring words are protected
  * replacement text copies the original colour, size, baseline and (best
    matching) font

Install
  pip install opencv-python-headless pytesseract pillow numpy pymupdf
  + the Tesseract binary:  Linux: sudo apt install tesseract-ocr
                           macOS: brew install tesseract
                           Windows: https://github.com/UB-Mannheim/tesseract/wiki
  + extra OCR languages (e.g. Hindi): sudo apt install tesseract-ocr-hin

Usage
  python textswap.py scan.jpg                       # interactive: asks what to do
  python textswap.py scan.jpg --find "John Smith" --replace "Amit Kumar"
  python textswap.py scan.jpg --find "4821 7733" --remove --all
  python textswap.py form.pdf --find Draft --remove --lang eng+hin
"""
from __future__ import annotations

# =============================================================================
# One-time runtime bootstrap (silent; installs only what is missing)
# =============================================================================
import glob as _glob
import importlib.util as _importlib_util
import os as _os
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys

_RUNTIME_NOTES = []
_RUNTIME_ERROR = ""
_TESSERACT_PATH = None


def _boot_msg(message):
    try:
        from IPython.display import HTML, display

        display(
            HTML(
                '<div style="font-family:Segoe UI,system-ui,Arial,sans-serif;'
                'padding:16px 18px;border-radius:14px;background:linear-gradient(135deg,#eef2ff,#faf5ff);'
                'border:1px solid #c7d2fe;color:#3730a3;font-size:13px;line-height:1.6">'
                + message
                + "</div>"
            )
        )
    except Exception:
        pass


def _run_shell(cmd, timeout=900):
    try:
        return _subprocess.run(
            cmd,
            shell=isinstance(cmd, str),
            stdout=_subprocess.PIPE,
            stderr=_subprocess.PIPE,
            timeout=timeout,
        )
    except Exception as exc:
        class _Result:
            returncode = 1
            stdout = b""
            stderr = str(exc).encode("utf-8", "ignore")

        return _Result()


def _missing_module(name):
    try:
        return _importlib_util.find_spec(name) is None
    except Exception:
        return True


def _pip_ensure(packages):
    if not packages:
        return True
    ok = False
    for extra in ([], ["--user"]):
        result = _run_shell(
            [
                _sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--disable-pip-version-check",
                *extra,
                *packages,
            ],
            timeout=900,
        )
        if result.returncode == 0:
            ok = True
            break
    if ok:
        _importlib_util.invalidate_caches()
    else:
        _RUNTIME_NOTES.append(
            "Kuch Python packages install nahi ho paaye. Runtime > Restart session try karein."
        )
    return ok


def _find_tesseract():
    found = _shutil.which("tesseract")
    if found:
        return found
    candidates = [
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        _os.path.join(_os.environ.get("LOCALAPPDATA", ""), "Programs", "Tesseract-OCR", "tesseract.exe"),
        _os.path.join(_os.environ.get("LOCALAPPDATA", ""), "Tesseract-OCR", "tesseract.exe"),
        "/usr/bin/tesseract",
        "/usr/local/bin/tesseract",
        "/opt/homebrew/bin/tesseract",
    ]
    for path in candidates:
        if path and _os.path.exists(path):
            return path
    return None


# ---- Python packages ----
_packages = []
if _missing_module("cv2"):
    _packages.append("opencv-python-headless")
if _missing_module("pytesseract"):
    _packages.append("pytesseract")
if _missing_module("PIL"):
    _packages.append("pillow")
if _missing_module("numpy"):
    _packages.append("numpy")
if _missing_module("IPython"):
    _packages.append("ipython")
if _missing_module("fitz") and _missing_module("pymupdf"):
    _packages.append("pymupdf")

if _packages:
    _boot_msg("⚙️ <b>Engine prepare ho raha hai…</b><br>Required Python packages install kiye ja rahe hain. First run me 1–2 minute lag sakte hain.")
    _pip_ensure(_packages)
else:
    _boot_msg("⚙️ <b>Engine prepare ho raha hai…</b><br>OCR engine check ho raha hai. First run me 1–2 minute lag sakte hain.")

# ---- Tesseract binary + language packs ----
_TESSERACT_PATH = _find_tesseract()
if not _TESSERACT_PATH and _os.name != "nt" and _shutil.which("apt-get"):
    _boot_msg("⚙️ <b>OCR engine install ho raha hai…</b><br>Tesseract + Hindi language pack install kiye ja rahe hain. Please wait.")
    _run_shell("apt-get update -qq", timeout=900)
    result = _run_shell(
        "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends "
        "tesseract-ocr tesseract-ocr-eng tesseract-ocr-hin",
        timeout=1200,
    )
    _TESSERACT_PATH = _find_tesseract()
    if not _TESSERACT_PATH:
        _RUNTIME_ERROR = (
            "Tesseract OCR install nahi ho paaya. Internet connection check karke notebook cell dobara run karein. "
            "Colab me: Runtime > Restart session."
        )
elif not _TESSERACT_PATH:
    _RUNTIME_ERROR = (
        "Tesseract OCR binary nahi mila. Windows par https://github.com/UB-Mannheim/tesseract/wiki se install karein, "
        "ya notebook ko Google Colab me chalayein."
    )

if _TESSERACT_PATH:
    try:
        import pytesseract as _pt

        _pt.pytesseract.tesseract_cmd = _TESSERACT_PATH
    except Exception as exc:
        _RUNTIME_ERROR = _RUNTIME_ERROR or f"pytesseract load nahi hua: {exc}"

# ---- Extra Hindi language pack + Devanagari fonts on Linux/Colab ----
if _RUNTIME_ERROR == "" and _os.name != "nt" and _shutil.which("apt-get"):
    try:
        import pytesseract as _pt

        installed = set(_pt.get_languages(config=""))
        if "hin" not in installed:
            _run_shell(
                "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends tesseract-ocr-hin",
                timeout=600,
            )
            _RUNTIME_NOTES.append("Hindi OCR pack install kiya gaya.")
        # Match the source document's typeface as closely as possible.  These are
        # metric-compatible clones of Arial/Times/Calibri/Cambria plus the Noto
        # and DejaVu families already used by Colab.
        font_pkgs = [
            "fonts-liberation",
            "fonts-crosextra-carlito",
            "fonts-crosextra-caladea",
            "fonts-dejavu-core",
            "fonts-roboto",
            "fonts-noto-core",
            "fonts-urw-base35",
        ]
        # Colab already ships some of these; install only what is actually
        # missing.  Checking just one package used to skip Carlito/Caladea/URW
        # on runtimes where Liberation was preinstalled, which weakened the
        # automatic font match for Calibri/Cambria documents.
        missing_fonts = [
            pkg for pkg in font_pkgs
            if _run_shell(f"dpkg -s {pkg}", timeout=45).returncode != 0
        ]
        if missing_fonts:
            _run_shell("apt-get update -qq", timeout=900)
            font_result = _run_shell(
                "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends "
                + " ".join(missing_fonts),
                timeout=900,
            )
            if font_result.returncode != 0:
                _RUNTIME_NOTES.append(
                    "Kuch document fonts install nahi hue; replacement text system font se match hoga."
                )
    except Exception:
        pass

# ---- Final OCR sanity check ----
if _RUNTIME_ERROR == "":
    try:
        import pytesseract as _pt

        _langs = _pt.get_languages(config="")
        if not _langs:
            raise RuntimeError("Tesseract ne koi language pack nahi bataya.")
    except Exception as exc:
        _RUNTIME_ERROR = (
            "Tesseract OCR ready nahi hai. " + str(exc) +
            " Notebook cell dobara run karein (setup automatic hai)."
        )

import argparse
import difflib
import math
import os
import re
import sys
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

try:
    import cv2
    import numpy as np
    import pytesseract
    from PIL import Image, ImageDraw, ImageFont, ImageOps
except Exception as _import_exc:
    _RUNTIME_ERROR = (
        f"OCR packages load nahi hue: {_import_exc}. "
        "Notebook cell dobara run karein (setup automatic hai)."
    )

# =============================================================================
# Image / PDF I/O
# =============================================================================


@dataclass
class Page:
    bgr: np.ndarray
    alpha: np.ndarray | None = None
    meta: dict = field(default_factory=dict)


def load_pages(path: str, dpi: int = 300) -> list[Page]:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        try:
            import pymupdf as fitz  # PyMuPDF >= 1.24
        except ImportError:
            import fitz

        pages = []
        with fitz.open(path) as doc:
            for p in doc:
                pix = p.get_pixmap(dpi=dpi, alpha=False)
                rgb = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, 3)
                pages.append(Page(rgb[:, :, ::-1].copy(), meta={"dpi": (dpi, dpi), "mode": "RGB"}))
        return pages

    im = Image.open(path)
    meta = {"dpi": im.info.get("dpi"), "icc": im.info.get("icc_profile"), "mode": im.mode}
    im = ImageOps.exif_transpose(im)  # pixels upright, so OCR coordinates make sense
    exif = im.getexif()
    meta["exif"] = exif if len(exif) else None
    alpha = None
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        alpha = np.asarray(im)[:, :, 3].copy()
    rgb = np.asarray(im.convert("RGB"))
    return [Page(rgb[:, :, ::-1].copy(), alpha, meta)]


def _to_pil(p: Page) -> Image.Image:
    im = Image.fromarray(p.bgr[:, :, ::-1])
    if p.alpha is not None:
        im = im.convert("RGBA")
        im.putalpha(Image.fromarray(p.alpha))
    elif p.meta.get("mode") == "L":
        im = im.convert("L")
    return im


def save_pages(pages: list[Page], out_path: str) -> None:
    ext = Path(out_path).suffix.lower()
    pils = [_to_pil(p) for p in pages]
    meta = pages[0].meta
    kw: dict = {}
    if ext == ".pdf":
        dpi = meta.get("dpi") or (300, 300)
        pils = [p.convert("RGB") for p in pils]
        pils[0].save(out_path, "PDF", save_all=True, append_images=pils[1:],
                     resolution=float(dpi[0]), quality=95)
        return
    if meta.get("dpi"):
        kw["dpi"] = meta["dpi"]
    if meta.get("icc"):
        kw["icc_profile"] = meta["icc"]
    if ext in (".jpg", ".jpeg"):
        kw.update(quality=97, subsampling=0)
        pils[0] = pils[0].convert("RGB")
    if meta.get("exif") is not None and ext in (".jpg", ".jpeg", ".png", ".webp"):
        kw["exif"] = meta["exif"]
    pils[0].save(out_path, **kw)


# =============================================================================
# OCR + text search
# =============================================================================


def _odd(n) -> int:
    n = int(n)
    return n if n % 2 else n + 1


@dataclass
class Word:
    text: str
    x: int
    y: int
    w: int
    h: int
    conf: float


@dataclass
class Match:
    page: int
    text: str            # exact OCR substring that matched
    box: tuple           # (x0, y0, x1, y1) approximate text box
    next_x: float | None  # left edge of the following word on the line
    score: float         # 1.0 = exact, <1 = fuzzy
    context: str         # whole OCR line with the match in [[ ]]


def _ocr_variants(gray: np.ndarray) -> list[np.ndarray]:
    """Extra views of the page that help OCR on textured / uneven backgrounds."""
    k = _odd(max(15, min(gray.shape) // 40))
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    def binarise(x):
        x = cv2.normalize(x, None, 0, 255, cv2.NORM_MINMAX)
        _, b = cv2.threshold(x, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return 255 - b                                  # dark text on white
    dark_on_light = binarise(cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, ker))
    light_on_dark = binarise(cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, ker))
    return [dark_on_light, light_on_dark]


def _tess(gray: np.ndarray, lang: str, psm: int, scale: float) -> list[list[Word]]:
    if scale != 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    d = pytesseract.image_to_data(gray, lang=lang, config=f"--psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    lines: dict = {}
    for i, raw in enumerate(d["text"]):
        t = raw.strip()
        if not t or float(d["conf"][i]) < 0:
            continue
        key = (d["page_num"][i], d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(Word(
            t, int(d["left"][i] / scale), int(d["top"][i] / scale),
            max(1, int(round(d["width"][i] / scale))), max(1, int(round(d["height"][i] / scale))),
            float(d["conf"][i])))
    out = [sorted(v, key=lambda k: k.x) for v in lines.values()]
    out.sort(key=lambda l: (l[0].y, l[0].x))
    return out


def ocr_lines(bgr: np.ndarray, lang: str = "eng", psm: int = 3, enhance: bool = False) -> list[list[Word]]:
    h, w = bgr.shape[:2]
    scale = 1.0 if max(h, w) >= 2000 else min(3.0, 2000.0 / max(h, w))
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if not enhance:
        return _tess(gray, lang, psm, scale)
    best, best_score = [], -1.0
    for g in [gray] + _ocr_variants(gray):               # keep the view the OCR understands best
        lines = _tess(g, lang, psm, scale)
        score = sum(wd.conf for ln in lines for wd in ln if wd.conf > 40)
        if score > best_score:
            best, best_score = lines, score
    return best


def _nfkc(s: str) -> str:
    return unicodedata.normalize("NFKC", s)


def _index_line(line: list[Word], case_sensitive: bool):
    raw = " ".join(_nfkc(w.text) for w in line)
    s = raw
    if not case_sensitive:
        low = raw.lower()
        s = low if len(low) == len(raw) else raw
    spans, pos = [], 0
    for w in line:
        n = len(_nfkc(w.text))
        spans.append((pos, pos + n))
        pos += n + 1
    return raw, s, spans


def _make_match(page_no, line, raw, spans, s0, e0, score) -> Match:
    idx = [i for i, (a, b) in enumerate(spans) if b > s0 and a < e0]
    fi, li = idx[0], idx[-1]
    wf, wl = line[fi], line[li]
    a, b = spans[fi]
    x0 = wf.x + wf.w * max(0, s0 - a) / max(1, b - a)
    a, b = spans[li]
    x1 = wl.x + wl.w * min(1.0, (e0 - a) / max(1, b - a))
    y0 = min(w.y for w in line[fi:li + 1])
    y1 = max(w.y + w.h for w in line[fi:li + 1])
    ends_word = e0 >= spans[li][1]
    if ends_word:
        next_x = line[li + 1].x if li + 1 < len(line) else None
    else:
        next_x = x1
    ctx = raw[:s0] + "[[" + raw[s0:e0] + "]]" + raw[e0:]
    return Match(page_no, raw[s0:e0], (x0, y0, x1, y1), next_x, score, ctx)


def find_matches(all_lines: list[list[list[Word]]], query: str, case_sensitive=False,
                 whole_word=False, fuzzy=0.8) -> list[Match]:
    q = " ".join(_nfkc(query).split())
    if not q:
        return []
    qs = q if case_sensitive else q.lower()
    pat = re.escape(qs)
    if whole_word:
        pat = r"(?<!\w)" + pat + r"(?!\w)"
    found: list[Match] = []
    for pno, lines in enumerate(all_lines):
        for line in lines:
            raw, s, spans = _index_line(line, case_sensitive)
            for m in re.finditer(pat, s):
                found.append(_make_match(pno, line, raw, spans, m.start(), m.end(), 1.0))
    if found or fuzzy <= 0:
        return found

    # fuzzy fallback: tolerate OCR slips such as O/0, l/1, rn/m
    nw = len(qs.split())
    for pno, lines in enumerate(all_lines):
        for line in lines:
            raw, s, spans = _index_line(line, case_sensitive)
            toks = [s[a:b] for a, b in spans]
            cands = []
            for i in range(len(toks)):
                for j in range(i, min(len(toks), i + nw + 1)):
                    if j - i + 1 < max(1, nw - 1):
                        continue
                    cand = " ".join(toks[i:j + 1]).strip(".,;:!?()[]\"'")
                    r = difflib.SequenceMatcher(None, cand, qs).ratio()
                    if r >= fuzzy:
                        cands.append((r, i, j))
            used: set = set()
            for r, i, j in sorted(cands, key=lambda c: -c[0]):
                if used & set(range(i, j + 1)):
                    continue
                used |= set(range(i, j + 1))
                found.append(_make_match(pno, line, raw, spans, spans[i][0], spans[j][1], r))
    return found


# =============================================================================
# Watermark / underlay detection
# =============================================================================

@dataclass
class UnderlayProfile:
    """A watermark, logo or stamp that shares the page with the matched text.

    ``mask`` is the visible underlay ink inside the current region, ``bridge``
    is an estimate of the same structure continued across the old glyphs and
    ``alpha`` is the per-pixel veil strength used when the watermark sits on
    top of the text.  All arrays are in region (ctx) coordinates.
    """
    present: bool = False
    order: str = "none"          # "under" | "over" | "none"
    confidence: float = 0.0
    coverage: float = 0.0
    mask: np.ndarray | None = None
    bridge: np.ndarray | None = None
    alpha: np.ndarray | None = None
    color: tuple = (0.0, 0.0, 0.0)
    kind: str = "watermark"


def detect_underlay_page(img: np.ndarray, sensitivity: float = 0.5,
                         max_side: int = 1400) -> np.ndarray:
    """Find low-contrast structured ink (watermark / logo / stamp) on a page.

    Normal text is high contrast and is deliberately excluded.  What remains
    is mid-tone, image-like ink: a grey crest, a coloured seal, a repeated
    diagonal word, a bank stamp.  Colour is as important as brightness -- on
    a tinted scan the watermark is often *neutral* while the paper is pink --
    so the score combines local luminance and local chroma deviation.
    """
    h0, w0 = img.shape[:2]
    if min(h0, w0) < 48:
        return np.zeros((h0, w0), bool)
    scale = min(1.0, float(max_side) / float(max(h0, w0)))
    if scale < 0.999:
        small = cv2.resize(img, (max(2, int(round(w0 * scale))), max(2, int(round(h0 * scale)))),
                           interpolation=cv2.INTER_AREA)
    else:
        small = img
    h, w = small.shape[:2]
    lab = cv2.cvtColor(small, cv2.COLOR_BGR2LAB).astype(np.float32)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    # A wide median keeps illumination gradients but rejects strokes thinner
    # than the kernel, so a crest/logo survives as a positive residual.
    k = _odd(np.clip(min(h, w) * 0.07, 25, 101))
    bg = cv2.medianBlur(lab.astype(np.uint8), int(k)).astype(np.float32)
    dl = bg[..., 0] - lab[..., 0]                  # >0: darker than local paper
    da = lab[..., 1] - bg[..., 1]
    db = lab[..., 2] - bg[..., 2]
    chroma = np.sqrt(da * da + db * db)
    score = 1.8 * chroma + 0.55 * np.maximum(dl, 0)
    score = cv2.GaussianBlur(score, (0, 0), 1.15)

    # Strong ink is content text, not underlay.  The absolute fallback catches
    # inverted scans where the local residual is not positive.
    strong = (np.abs(dl) > 58) | (gray < 105)
    near_text = cv2.dilate(strong.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
    vals = score[~near_text]
    if vals.size < 50:
        return np.zeros((h0, w0), bool)
    med = float(np.median(vals))
    mad = 1.4826 * float(np.median(np.abs(vals - med)))
    base = max(4.0, med + 2.6 * mad)
    sens = float(np.clip(sensitivity, 0.0, 1.0))
    thr = max(3.2, base * (1.30 - 0.60 * sens))
    cand = (score > thr) & ~near_text
    cand &= (dl > 2.0) | (chroma > 2.0)
    u8 = cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    u8 = cv2.morphologyEx(u8, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(u8, 8)
    keep = np.zeros_like(u8)
    min_area = max(8, int(round(h * w * 0.000006)))
    for i in range(1, n):
        x, y, ww, hh, area = stats[i]
        if area >= min_area and (max(ww, hh) >= 7 or area >= 30):
            keep[labels == i] = 1
    if scale < 0.999:
        keep = cv2.resize(keep, (w0, h0), interpolation=cv2.INTER_NEAREST)
        keep = cv2.dilate(keep, np.ones((3, 3), np.uint8)) > 0
    return keep


def _build_underlay_profile(ctx: np.ndarray, target: np.ndarray, core: np.ndarray,
                            mask: np.ndarray, roi: tuple, bh: int, stroke: float,
                            underlay: np.ndarray, color: tuple,
                            forced: str = "auto") -> UnderlayProfile:
    """Decide whether a local underlay is present and how it layers with text."""
    prof = UnderlayProfile()
    H, W = target.shape
    m = underlay.astype(bool).copy()
    if not m.any():
        return prof
    # Text anti-alias fringes look like low-contrast ink; never let them become
    # "watermark" pixels that would survive the erase and leave a ghost.
    near = cv2.dilate(target.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
    m &= ~near
    if m.sum() < 6:
        return prof

    rl, rt, rr, rb = roi
    ex = max(4, int(round(bh * 1.1)))
    er0, er1 = max(0, rt - ex), min(H, rb + ex)
    ec0, ec1 = max(0, rl - ex), min(W, rr + ex)
    zone = np.zeros((H, W), bool)
    zone[er0:er1, ec0:ec1] = True
    hits = m & zone
    zone_area = max(1, int(zone.sum()))
    cov = float(hits.sum()) / float(zone_area)
    if hits.sum() < max(8, int(round(zone_area * 0.004))):
        return prof

    n, labels, stats, _ = cv2.connectedComponentsWithStats(hits.astype(np.uint8), 8)
    comps = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= 4]
    if not comps:
        return prof

    prof.present = True
    prof.coverage = round(min(1.0, cov * 8.0), 3)
    # Continue the visible watermark across the text only where the old glyphs
    # are.  Closing with a kernel wider than a stem bridges the interruption.
    k = _odd(int(np.clip(stroke * 2.0 + 5.0, 5, 31)))
    closed = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))) > 0
    hidden = closed & mask

    g = cv2.cvtColor(ctx, cv2.COLOR_BGR2GRAY)
    ink_lum = 0.114 * color[0] + 0.587 * color[1] + 0.299 * color[2]
    probe = hidden & target
    light_frac = 0.0
    if probe.sum() >= 5:
        light_frac = float(np.mean(g[probe] > ink_lum + 24.0))

    order = "under"
    z_conf = 0.58
    if probe.sum() >= 5:
        if light_frac >= 0.34:
            order = "over"
            z_conf = float(np.clip(0.58 + light_frac, 0.0, 0.98))
        else:
            z_conf = float(np.clip(0.66 + (0.30 - min(0.30, light_frac)), 0.0, 0.95))
    if forced in ("under", "over"):
        order = forced
        z_conf = max(z_conf, 0.62)

    # Estimate the watermark colour and veil alpha from visible pixels only.
    vals = ctx[m].astype(np.float32)
    gv = g[m].astype(np.float32)
    if vals.size:
        cut = float(np.percentile(gv, 35))
        chosen = vals[gv <= cut]
        wcol = np.median(chosen if chosen.size else vals, axis=0).astype(np.float32)
    else:
        wcol = np.array(color, np.float32)
    known_paper = ~(mask | m | near)
    wgt = known_paper.astype(np.float32)
    paper = _norm_blur(ctx.astype(np.float32), wgt, max(2.0, bh * 0.6)).astype(np.float32)
    den = paper - wcol
    den2 = np.sum(den * den, axis=2)
    alpha = np.zeros((H, W), np.float32)
    if float(den2[m].mean()) > 1.0:
        a_raw = np.sum((paper - ctx.astype(np.float32)) * den, axis=2) / np.maximum(den2, 1.0)
        alpha[m] = np.clip(a_raw[m], 0.0, 1.0)
        if hidden.any():
            a8 = np.clip(alpha * 255.0, 0, 255).astype(np.uint8)
            fill8 = cv2.inpaint(a8, mask.astype(np.uint8) * 255, 2, cv2.INPAINT_TELEA)
            alpha[hidden] = (fill8[hidden].astype(np.float32) / 255.0)

    conf = 0.45 * min(1.0, cov / 0.06) + 0.25 * min(1.0, len(comps) / 3.0) + 0.30 * z_conf
    prof.order = order
    prof.confidence = round(float(np.clip(conf, 0.25, 0.99)), 3)
    prof.mask = m
    prof.bridge = hidden
    prof.alpha = alpha
    prof.color = tuple(float(v) for v in wcol)
    big = any(stats[i, cv2.CC_STAT_WIDTH] > W * 0.45 or stats[i, cv2.CC_STAT_AREA] > H * W * 0.02
              for i in comps)
    prof.kind = "logo" if big else "watermark"
    return prof


def _underlay_report(prof: UnderlayProfile | None, mode: str = "auto") -> dict:
    if prof is None or not prof.present:
        return {"present": False, "order": "none", "confidence": 0.0,
                "coverage": 0.0, "kind": "none", "mode": mode}
    return {
        "present": True,
        "order": prof.order,
        "confidence": round(float(prof.confidence), 3),
        "coverage": round(float(prof.coverage), 3),
        "kind": prof.kind,
        "mode": mode,
    }


# =============================================================================
# Pixel-level analysis: find the ink of the matched text
# =============================================================================


@dataclass
class Region:
    ctx: tuple             # (x0, y0, x1, y1) working window in page coordinates
    target: np.ndarray     # bool, ink of the matched text
    mask: np.ndarray       # bool, pixels that will be rebuilt (dilated target)
    protect: np.ndarray    # bool, ink that must NOT be touched (neighbours, lines)
    ink_bbox: tuple        # tight bbox of the target ink incl. anti-alias fringe
    core: np.ndarray       # bool, half-coverage contour (used for font/size/baseline fitting)
    core_bbox: tuple       # tight bbox of the core (page coordinates)
    color: tuple           # BGR ink colour
    bh: int                # nominal text height
    roi: tuple = (0, 0, 0, 0)  # padded match box in ctx coordinates (l, t, r, b)
    soften: float = 0.5    # edge softness estimated from the source ink
    stroke: float = 2.0    # typical full stroke width in pixels
    underlay: UnderlayProfile | None = None


def analyze_region(img: np.ndarray, box: tuple, underlay_mask: np.ndarray | None = None,
                   watermark_mode: str = "auto", watermark_sensitivity: float = 0.5) -> Region | None:
    H, W = img.shape[:2]
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    bh = max(4, y1 - y0)
    pad = max(2, int(round(bh * 0.12)))
    margin = max(6, int(round(bh * 1.2)))
    rx0, ry0, rx1, ry1 = max(0, x0 - pad), max(0, y0 - pad), min(W, x1 + pad), min(H, y1 + pad)
    cx0, cy0, cx1, cy1 = max(0, rx0 - margin), max(0, ry0 - margin), min(W, rx1 + margin), min(H, ry1 + margin)
    ctx = img[cy0:cy1, cx0:cx1]

    # local background estimate (handles gradients, shadows, coloured paper)
    bg = cv2.medianBlur(ctx, _odd(np.clip(bh * 3, 21, 201)))
    lab = cv2.cvtColor(ctx, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab_bg = cv2.cvtColor(bg, cv2.COLOR_BGR2LAB).astype(np.float32)
    dist = np.linalg.norm(lab - lab_bg, axis=2)

    roi = (slice(ry0 - cy0, ry1 - cy0), slice(rx0 - cx0, rx1 - cx0))
    d8 = np.clip(dist[roi], 0, 255).astype(np.uint8)
    otsu, _ = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thr = max(18.0, 0.45 * otsu)
    ink = dist > thr

    # long thin structures (underlines, table borders) are not part of the text
    L = max(25, int(bh * 3))
    u8 = ink.astype(np.uint8)
    hl = cv2.morphologyEx(u8, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (L, 1)))
    vl = cv2.morphologyEx(u8, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, L)))
    lines = cv2.dilate(hl | vl, np.ones((3, 3), np.uint8)) > 0

    n, labels, stats, _ = cv2.connectedComponentsWithStats((ink & ~lines).astype(np.uint8), connectivity=8)
    rl, rt, rr, rb = rx0 - cx0, ry0 - cy0, rx1 - cx0, ry1 - cy0
    # Separate high-contrast content text from mid-tone underlay ink.  A
    # watermark arc crossing the OCR box must not be treated as a glyph: that
    # is what used to erase the crest behind the name.
    level = float(np.percentile(dist[ink], 95)) if ink.any() else float(thr)
    text_thr = max(float(thr), 0.55 * float(otsu), 0.45 * level)
    target = np.zeros(ink.shape, bool)
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        ix = max(0, min(x + w, rr) - max(x, rl))
        iy = max(0, min(y + h, rb) - max(y, rt))
        if area < 2 or ix * iy < 0.6 * w * h:
            continue
        comp = labels == i
        strength = float(np.percentile(dist[comp], 75))
        if strength >= text_thr:               # glyph, not a faint crest stroke
            target |= comp
    # A dotted underline is not part of the word, but its dots are strong ink
    # and can sit inside a wide OCR box.  Left in, they inflate the measured
    # glyph height and the replacement comes out far too large.
    tiny_h = max(3, int(round(bh * 0.22)))
    row = {}
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if h > tiny_h or w > max(7, int(round(bh * 0.40))) or area > max(20, bh * bh * 0.10):
            continue
        yc = int(round((y + h * 0.5) / 2.0)) * 2
        row.setdefault(yc, []).append(i)
    dot_line = np.zeros(ink.shape, bool)
    for yc, ids in row.items():
        if len(ids) < 6:
            continue
        xs0 = min(stats[i, cv2.CC_STAT_LEFT] for i in ids)
        xs1 = max(stats[i, cv2.CC_STAT_LEFT] + stats[i, cv2.CC_STAT_WIDTH] for i in ids)
        if xs1 - xs0 >= 0.5 * max(1, x1 - x0):
            for i in ids:
                dot_line |= labels == i
    if dot_line.any():
        target &= ~dot_line
        lines = lines | dot_line
    # Strong ink inside the OCR box itself belongs to the match.  Faint ink
    # inside the same box is left alone, not blanket-added.
    bl, bt, br, bb = x0 - cx0, y0 - cy0, x1 - cx0, y1 - cy0
    inner = np.zeros(ink.shape, bool)
    inner[max(0, bt):max(0, bb), max(0, bl):max(0, br)] = True
    target |= (ink & ~lines) & inner & (dist >= text_thr)
    if not target.any():
        return None

    r = max(1, int(round(bh / 20)))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    other = (ink & ~target) | lines
    mask = (cv2.dilate(target.astype(np.uint8), k) > 0) & ~other

    ys, xs = np.nonzero(target)
    ink_bbox = (cx0 + int(xs.min()), cy0 + int(ys.min()), cx0 + int(xs.max()) + 1, cy0 + int(ys.max()) + 1)
    dt = dist[target]
    level = float(np.percentile(dt, 90))                  # typical full-ink contrast
    core = target & (dist >= 0.5 * level)                 # ~50% coverage contour
    if core.sum() < 4:
        core = target
    cys, cxs = np.nonzero(core)
    core_bbox = (cx0 + int(cxs.min()), cy0 + int(cys.min()), cx0 + int(cxs.max()) + 1, cy0 + int(cys.max()) + 1)
    # Use only the strongest ink pixels for colour.  The 80th percentile still
    # contains anti-aliased fringe and makes dark text look grey.
    strong = target & (dist >= np.percentile(dt, 92))
    if strong.sum() < 6:
        strong = core
    color = tuple(float(c) for c in np.median(ctx[strong], axis=0))

    # Estimate how soft the source glyph edges are, so freshly rendered text
    # gets the same amount of scan / anti-alias blur instead of looking pasted.
    fringe = target & ~core
    soften = 0.45
    if fringe.sum() > 8:
        from_core = cv2.distanceTransform((~core).astype(np.uint8), cv2.DIST_L2, 3)
        edge_w = float(np.percentile(from_core[fringe], 65))
        soften = float(np.clip(edge_w * 0.55, 0.25, 1.6))
    # A fixed pixel blur that is invisible on a 90 px heading destroys a 10 px
    # caption: every stem becomes a half-covered edge and the ink reads grey.
    # Scale the blur by glyph height so small replacements stay as dark and
    # crisp as the source they replace.
    soften *= min(1.0, max(0.0, bh / 60.0))

    # Typical full stroke width.  Distance transform on the core gives the
    # half-width at each interior pixel; the 85th percentile tracks the stem.
    dt_core = cv2.distanceTransform(core.astype(np.uint8), cv2.DIST_L2, 5)
    stroke = float(max(1.0, np.percentile(dt_core[core], 85) * 2.0))

    # Underlay-aware protection.  Visible watermark pixels are added to
    # ``protect`` so the erase mask never covers them, and the local profile
    # tells erase/draw how to rebuild the hidden continuation.
    underlay_prof = None
    if (underlay_mask is not None and watermark_mode != "off"
            and getattr(underlay_mask, "shape", None) == img.shape[:2]):
        crop = underlay_mask[cy0:cy1, cx0:cx1].astype(bool)
        if crop.any():
            near_t = cv2.dilate(target.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
            safe = crop & ~near_t
            if safe.any():
                other = other | safe
                mask = (cv2.dilate(target.astype(np.uint8), k) > 0) & ~other
                underlay_prof = _build_underlay_profile(
                    ctx, target, core, mask, (rl, rt, rr, rb), bh, stroke,
                    safe, color, forced=watermark_mode,
                )

    return Region((cx0, cy0, cx1, cy1), target, mask, other, ink_bbox, core, core_bbox,
                  color, bh, (rl, rt, rr, rb), soften, stroke, underlay_prof)


# =============================================================================
# Erase: rebuild the background
# =============================================================================


def _norm_blur(img: np.ndarray, w: np.ndarray, sigma: float) -> np.ndarray:
    """Gaussian average of only the 'known' pixels (weights w), multi-scale."""
    out = np.zeros_like(img)
    done = np.zeros(img.shape[:2], bool)
    for s in (sigma, sigma * 2, sigma * 4, sigma * 8):
        num = cv2.GaussianBlur(img * w[..., None], (0, 0), s)
        den = cv2.GaussianBlur(w, (0, 0), s)
        ok = (den > 0.06) & ~done
        out[ok] = num[ok] / den[ok][:, None]
        done |= ok
    if not done.all():
        out[~done] = img[w > 0].mean(0) if (w > 0).any() else 255
    return out


def _robust_std(res: np.ndarray) -> np.ndarray:
    if res.size == 0:
        return np.zeros(3, np.float32)
    med = np.median(res, axis=0)
    return 1.4826 * np.median(np.abs(res - med), axis=0)


def erase(out: np.ndarray, src: np.ndarray, reg: Region, method: str = "auto", seed: int = 0) -> str:
    cx0, cy0, cx1, cy1 = reg.ctx
    ctx = src[cy0:cy1, cx0:cx1].astype(np.float32)
    prot = cv2.dilate(reg.protect.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    known = ~reg.mask & ~prot
    # sample only pixels of the SAME surface as the text sits on (not the paper next to a coloured band)
    rl, rt, rr, rb = reg.roi
    labc = cv2.cvtColor(ctx.astype(np.uint8), cv2.COLOR_BGR2LAB).astype(np.float32)
    rk = known[rt:rb, rl:rr]
    if rk.sum() > 20:
        ref = np.median(labc[rt:rb, rl:rr][rk], axis=0)
        known &= np.linalg.norm(labc - ref, axis=2) < 45
    wgt = known.astype(np.float32)

    under = reg.underlay
    if under is not None and under.present and under.order == "under":
        # Flat fill creates a rectangular smear over a crest/stamp.  Rebuild
        # only the old glyph strokes and let the surrounding watermark ink
        # continue through them, then add a conservative structural hint from
        # the underlay alpha map.
        used = "watermark"
        rad = int(np.clip(round(reg.stroke * 0.45), 2, 4))
        fill = cv2.inpaint(ctx.astype(np.uint8), reg.mask.astype(np.uint8) * 255, rad,
                           cv2.INPAINT_TELEA).astype(np.float32)
        if under.bridge is not None and under.alpha is not None and under.bridge.any():
            paper_known = ~(reg.mask | under.mask | prot)
            paper = _norm_blur(ctx, paper_known.astype(np.float32), max(2.0, reg.bh * 0.6))
            wcol = np.array(under.color, np.float32)
            au = np.clip(under.alpha, 0.0, 1.0)
            struct = paper * (1 - au[..., None]) + wcol * au[..., None]
            wstruct = under.bridge.astype(np.float32) * 0.55 * np.clip(au * 3.0, 0.0, 1.0)
            wstruct *= float(np.clip(under.confidence, 0.3, 0.95))
            fill = fill * (1 - wstruct[..., None]) + struct * wstruct[..., None]
        m = reg.mask.astype(np.float32)
        grow = cv2.dilate(reg.mask.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(np.float32)
        a = np.maximum(m, cv2.GaussianBlur(m, (0, 0), 0.7) * grow)
        a[reg.protect] = 0
        base = out[cy0:cy1, cx0:cx1].astype(np.float32)
        out[cy0:cy1, cx0:cx1] = np.clip(base * (1 - a[..., None]) + fill * a[..., None], 0, 255).astype(np.uint8)
        return used

    used = method
    if known.sum() < 20:
        used = "inpaint"
    if used == "auto":
        low = _norm_blur(ctx, wgt, max(3.0, reg.bh * 0.6))
        std = _robust_std((ctx - low)[known])
        used = "inpaint" if float(std.mean()) > 9.0 else "fill"

    if used == "fill":
        low = _norm_blur(ctx, wgt, max(3.0, reg.bh * 0.6))
        std = _robust_std((ctx - low)[known])             # paper grain per channel
        fill = _norm_blur(ctx, wgt, max(2.0, reg.bh * 0.30))
        rng = np.random.default_rng(seed)
        noise = cv2.GaussianBlur(rng.standard_normal(ctx.shape).astype(np.float32), (0, 0), 0.6)
        noise /= max(float(noise.std()), 1e-6)
        fill = np.clip(fill + noise * std, 0, 255)
    else:
        fill = cv2.inpaint(ctx.astype(np.uint8), reg.mask.astype(np.uint8) * 255, 3,
                           cv2.INPAINT_TELEA).astype(np.float32)

    m = reg.mask.astype(np.float32)
    grow = cv2.dilate(reg.mask.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(np.float32)
    a = np.maximum(m, cv2.GaussianBlur(m, (0, 0), 0.7) * grow)
    a[reg.protect] = 0
    base = out[cy0:cy1, cx0:cx1].astype(np.float32)
    out[cy0:cy1, cx0:cx1] = np.clip(base * (1 - a[..., None]) + fill * a[..., None], 0, 255).astype(np.uint8)
    return used


# =============================================================================
# Replace: font matching + rendering
# =============================================================================

FONT_DIRS = ["/usr/share/fonts", "/usr/local/share/fonts", "~/.fonts", "~/.local/share/fonts",
             "/System/Library/Fonts", "/System/Library/Fonts/Supplemental", "/Library/Fonts",
             "~/Library/Fonts", "C:/Windows/Fonts"]
PREFERRED = ("arial", "helvet", "liberation", "dejavu", "times", "georgia", "calibri", "carlito",
             "cambria", "caladea", "verdana", "tahoma", "trebuc", "cour", "roboto", "opensans",
             "notosans", "notoserif", "segoe", "garamond", "palatino", "bookman", "century")


def discover_fonts(user_fonts: list[str] | None) -> list[str]:
    if user_fonts:
        found = []
        for f in user_fonts:
            f = os.path.expanduser(f)
            if os.path.isdir(f):
                for root, _, files in os.walk(f):
                    found += [os.path.join(root, x) for x in files if x.lower().endswith((".ttf", ".otf", ".ttc"))]
            else:
                found.append(f)
        return sorted(found)
    allf = []
    for d in FONT_DIRS:
        d = os.path.expanduser(d)
        if os.path.isdir(d):
            for root, _, files in os.walk(d):
                allf += [os.path.join(root, x) for x in files if x.lower().endswith((".ttf", ".otf"))]

    def rank(p):
        n = os.path.basename(p).lower()
        for i, k in enumerate(PREFERRED):
            if k in n:
                return i
        return 999

    ranked = sorted(allf, key=lambda p: (rank(p), p))
    pref = [p for p in ranked if rank(p) < 999]
    return (pref or ranked)[:60]


def has_glyphs(font_path: str, text: str) -> bool:
    try:
        f = ImageFont.truetype(font_path, 32)
        notdef = f.getmask("\uffff")
        nd = (notdef.size, bytes(notdef))
        for ch in set(text):
            if ch.isspace():
                continue
            m = f.getmask(ch)
            if (m.size, bytes(m)) == nd:
                return False
        return True
    except Exception:
        return False


def render_alpha(text: str, font_path: str, size: float, fx: float = 0.0, fy: float = 0.0, ss: int = 4):
    """Anti-aliased coverage map. Returns (alpha[0..1], L, T): alpha[0,0] sits at
    (origin_x + L, origin_y + T) where origin = left end of the baseline."""
    font = ImageFont.truetype(font_path, max(2, int(round(size * ss))))
    l, t, r, b = font.getbbox(text, anchor="ls")
    L, T = math.floor(l / ss) - 2, math.floor(t / ss) - 2
    R, B = math.ceil(r / ss) + 3, math.ceil(b / ss) + 3
    Wc, Hc = R - L, B - T
    big = Image.new("L", (Wc * ss, Hc * ss), 0)
    ImageDraw.Draw(big).text(((-L + fx) * ss, (-T + fy) * ss), text, font=font, fill=255, anchor="ls")
    a = np.asarray(big, dtype=np.float32)
    if ss > 1:
        a = cv2.resize(a, (Wc, Hc), interpolation=cv2.INTER_AREA)
    return a / 255.0, L, T


def _extent(a: np.ndarray, thr: float = 0.5):
    ys, xs = np.nonzero(a > thr)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def _align_to_mask(alpha: np.ndarray, L: float, T: float, ix: float, iy: float,
                   obs: np.ndarray, page_x0: int, page_y0: int,
                   max_shift: float = 2.0, step: float = 0.5) -> tuple[float, float]:
    """Find a small sub-pixel translation that best overlaps the source ink.

    Font hinting can move a glyph by a pixel even when its size is correct.
    Aligning the rendered source word to the observed core mask before placing
    the replacement removes that residual drift.  The search is deliberately
    tiny (a couple of pixels); it is a calibration, not a free-form layout.
    """
    oh, ow = obs.shape
    m = int(math.ceil(max_shift)) + 4
    bh, bw = oh + 2 * m, ow + 2 * m
    canvas = np.zeros((bh, bw), np.float32)
    # Keep both the baseline-origin and scaled-L fractions in the warp.  Array
    # placement is integer, but source calibration must not truncate subpixels.
    base_ix, base_iy = math.floor(ix), math.floor(iy)
    px = base_ix + L - page_x0 + m
    py = base_iy + T - page_y0 + m
    canvas_px, canvas_py = math.floor(px), math.floor(py)
    frac_x = (ix - base_ix) + (px - canvas_px)
    frac_y = (iy - base_iy) + (py - canvas_py)
    sx0, sy0 = max(0, -canvas_px), max(0, -canvas_py)
    dx0, dy0 = max(0, canvas_px), max(0, canvas_py)
    sw = min(alpha.shape[1] - sx0, bw - dx0)
    sh = min(alpha.shape[0] - sy0, bh - dy0)
    if sw <= 0 or sh <= 0:
        return 0.0, 0.0
    canvas[dy0:dy0 + sh, dx0:dx0 + sw] = alpha[sy0:sy0 + sh, sx0:sx0 + sw]

    best_dx = best_dy = 0.0
    best = -1.0
    steps = np.arange(-max_shift, max_shift + 1e-6, step)
    for dy in steps:
        for dx in steps:
            warp_x, warp_y = dx + frac_x, dy + frac_y
            if abs(warp_x) < 1e-9 and abs(warp_y) < 1e-9:
                shifted = canvas
            else:
                M = np.float32([[1, 0, warp_x], [0, 1, warp_y]])
                shifted = cv2.warpAffine(canvas, M, (bw, bh), flags=cv2.INTER_LINEAR)
            crop = shifted[m:m + oh, m:m + ow] > 0.5
            inter = np.logical_and(crop, obs).sum()
            union = np.logical_or(crop, obs).sum()
            iou = inter / float(max(1, union))
            if iou > best:
                best = iou
                best_dx, best_dy = float(dx), float(dy)
    return best_dx, best_dy



@dataclass
class FontFit:
    path: str
    size: float
    score: float
    weight: float = 0.0
    stretch: float = 1.0


_FONT_NAMES = {
    "arial": "Arial", "arialbd": "Arial Bold", "ariali": "Arial Italic",
    "arialbi": "Arial Bold Italic", "arialn": "Arial Narrow",
    "arialnb": "Arial Narrow Bold", "arialni": "Arial Narrow Italic",
    "arialnbi": "Arial Narrow Bold Italic", "arialuni": "Arial Unicode MS",
    "times": "Times New Roman", "timesbd": "Times New Roman Bold",
    "timesi": "Times New Roman Italic", "timesbi": "Times New Roman Bold Italic",
    "cour": "Courier New", "courbd": "Courier New Bold",
    "couri": "Courier New Italic", "courbi": "Courier New Bold Italic",
    "georgia": "Georgia", "georgiab": "Georgia Bold",
    "georgiai": "Georgia Italic", "georgiaz": "Georgia Bold Italic",
    "calibri": "Calibri", "calibrib": "Calibri Bold",
    "calibrii": "Calibri Italic", "calibriz": "Calibri Bold Italic",
    "calibril": "Calibri Light",
    "cambria": "Cambria", "cambriab": "Cambria Bold",
    "cambriai": "Cambria Italic", "cambriaz": "Cambria Bold Italic",
    "verdana": "Verdana", "verdanab": "Verdana Bold",
    "verdanai": "Verdana Italic", "verdanaz": "Verdana Bold Italic",
    "tahoma": "Tahoma", "tahomabd": "Tahoma Bold",
    "trebuc": "Trebuchet MS", "trebucbd": "Trebuchet MS Bold",
    "trebucit": "Trebuchet MS Italic", "trebucbi": "Trebuchet MS Bold Italic",
    "segoeui": "Segoe UI", "segoeuib": "Segoe UI Bold",
    "segoeuii": "Segoe UI Italic", "segoeuiz": "Segoe UI Bold Italic",
    "roboto-regular": "Roboto", "roboto-bold": "Roboto Bold",
    "roboto-italic": "Roboto Italic", "roboto-bolditalic": "Roboto Bold Italic",
    "liberationsans-regular": "Liberation Sans",
    "liberationsans-bold": "Liberation Sans Bold",
    "liberationsans-italic": "Liberation Sans Italic",
    "liberationsans-bolditalic": "Liberation Sans Bold Italic",
    "liberationserif-regular": "Liberation Serif",
    "liberationserif-bold": "Liberation Serif Bold",
    "liberationserif-italic": "Liberation Serif Italic",
    "liberationserif-bolditalic": "Liberation Serif Bold Italic",
    "dejavusans": "DejaVu Sans", "dejavusans-bold": "DejaVu Sans Bold",
    "dejavusans-oblique": "DejaVu Sans Oblique",
    "dejavusans-boldoblique": "DejaVu Sans Bold Oblique",
    "dejavuserif": "DejaVu Serif", "dejavuserif-bold": "DejaVu Serif Bold",
    "notosans-regular": "Noto Sans", "notosans-bold": "Noto Sans Bold",
    "notoserif-regular": "Noto Serif", "notoserif-bold": "Noto Serif Bold",
    "carlito-regular": "Carlito", "carlito-bold": "Carlito Bold",
    "carlito-italic": "Carlito Italic", "carlito-bolditalic": "Carlito Bold Italic",
    "caladea-regular": "Caladea", "caladea-bold": "Caladea Bold",
    "caladea-italic": "Caladea Italic", "caladea-bolditalic": "Caladea Bold Italic",
}


def font_label(font_path: str) -> str:
    stem = Path(font_path).stem.lower()
    if stem in _FONT_NAMES:
        return _FONT_NAMES[stem]
    # Common GNU/Linux naming with hyphens and weights.
    clean = re.sub(r"[_-]+", " ", stem)
    clean = re.sub(r"\b(regular|book|roman)\b", "", clean, flags=re.I)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean.title() or Path(font_path).name


def _stroke_width(mask: np.ndarray) -> float:
    m = np.asarray(mask)
    if m.dtype != np.bool_:
        # Alpha coverage maps are float 0..1; binary masks may be 0/255.
        m = (m > 0.5) if (m.size and float(m.max()) <= 1.0) else (m > 0)
    if not m.any():
        return 0.0
    dt = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 5)
    return float(max(0.0, np.percentile(dt[m], 85) * 2.0))


def _mask_score(rendered: np.ndarray, obs: np.ndarray) -> float:
    """Shape similarity plus a natural-aspect-ratio penalty.

    Both masks are compared at the same box size (so a single-pixel OCR height
    difference does not shift every later glyph), while the font's natural
    aspect ratio must still match the source.  That keeps Calibri from being
    beaten by a visually similar but wider/narrower face.
    """
    oh, ow = obs.shape
    rh, rw = rendered.shape
    if rh < 1 or rw < 1:
        return -1.0
    res = cv2.resize(rendered.astype(np.float32), (ow, oh),
                     interpolation=cv2.INTER_AREA) > 0.5
    inter = np.logical_and(res, obs).sum()
    union = np.logical_or(res, obs).sum()
    if not union:
        return -1.0
    iou = inter / union
    aspect_penalty = math.exp(-2.0 * abs(math.log((rw / rh) / (ow / oh))))
    return float(iou * aspect_penalty)


def _size_for_height(text: str, font_path: str, target_h: float, target_w: float | None = None) -> float:
    """Solve the raster size from observed ink height, then natural line width.

    Several nearby point sizes can have the same hinted height but different
    glyph advances.  When width is available, choose the size in that plateau
    whose natural whole-line width is closest instead of immediately hiding
    the discrepancy with horizontal stretching.  Measure with the same
    supersampling used by the final renderer so their hinting decisions agree.
    """
    a, _, _ = render_alpha(text, font_path, 100, ss=1)
    e = _extent(a)
    if not e or e[3] - e[1] < 1:
        return max(4.0, target_h * 0.9)
    size = max(4.0, 100.0 * target_h / (e[3] - e[1]))
    best_size = size
    best_error = (float("inf"), float("inf"))
    for _ in range(7):
        a, _, _ = render_alpha(text, font_path, size, ss=1 if size < 15 else 4)
        e = _extent(a)
        if not e:
            break
        h, w = e[3] - e[1], e[2] - e[0]
        error = (abs(h - target_h), abs(w - target_w) if target_w is not None else 0.0)
        if error < best_error:
            best_size, best_error = size, error
        height_done = abs(h - target_h) <= 0.35
        width_done = target_w is None or abs(w - target_w) <= 0.35
        if height_done and width_done:
            best_size = size
            break
        ratio = target_w / max(0.5, w) if height_done and target_w is not None else target_h / max(0.5, h)
        size = max(4.0, size * ratio)
    return best_size


def _font_prior(font_path: str) -> float:
    """Tiny tie-breaker that prefers the canonical face over lookalikes.

    Arial Unicode MS and Arial have nearly identical Latin glyphs.  When their
    scores are within noise, the plain Arial file is the better explanation of
    a normal document.  Symbol/emoji fonts should never win on glyph overlap.
    A clean scan also makes regular text look slightly heavier, so bold faces
    need a clearly better shape score before they are selected.
    """
    stem = Path(font_path).stem.lower()
    if any(k in stem for k in ("symbol", "wingding", "webding", "emoji", "icon", "dingbat")):
        return 0.55
    prior = 1.0
    if "unicode" in stem or stem == "arialuni":
        prior *= 0.95
    if any(k in stem for k in ("narrow", "condensed")):
        prior *= 0.96
    # A blurred/low-resolution regular glyph often has the same ink thickness
    # as a clean bold glyph.  Keep the regular face unless bold is clearly
    # better at reproducing the actual letter shapes.  Some Windows faces use
    # terse suffixes (calibrib, georgiab, calibriz); the label table knows them.
    label = _FONT_NAMES.get(stem, "").lower()
    if "bold" in label or any(
        k in stem for k in ("bold", "black", "heavy", "semibold", "demi")
    ):
        prior *= 0.92
    return prior



def _weighted_crop(rendered: np.ndarray, delta: float) -> np.ndarray:
    if abs(delta) < 0.04:
        return rendered
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    if delta > 0:
        changed = cv2.dilate(rendered, k)
    else:
        changed = cv2.erode(rendered, k)
    f = float(min(1.0, abs(delta)))
    return rendered * (1.0 - f) + changed * f


def fit_font(candidates: list[str], text: str, obs: np.ndarray) -> FontFit:
    """Pick the installed font that reproduces the observed glyph shapes.

    The renderer keeps the source aspect ratio: only height is normalised when
    scoring, so a narrow or wide font cannot win merely by being stretched into
    the OCR box.  A small residual stroke-weight correction is measured after
    the family/style has been chosen.
    """
    oh, ow = obs.shape
    best = FontFit(candidates[0] if candidates else "", max(4.0, oh * 0.9), 0.0)
    for path in candidates:
        try:
            a, _, _ = render_alpha(text, path, 100, ss=1)
            e = _extent(a)
            if not e or e[3] - e[1] < 1:
                continue
            size = _size_for_height(text, path, oh, ow)
            a, _, _ = render_alpha(text, path, size, ss=1 if size < 15 else 4)
            e = _extent(a)
            if not e:
                continue
            crop = a[e[1]:e[3], e[0]:e[2]]
            score = _mask_score(crop, obs) * _font_prior(path)
            if score > best.score:
                best = FontFit(path, size, score)
        except Exception:
            continue
    if not best.path:
        return best

    # Height is locked; width is matched with a pure horizontal stretch below.
    # Changing the point size here (the old behaviour) also changed glyph
    # height by a pixel, which is exactly what pixel-lock must never do.

    # Estimate the residual stroke difference.  This catches a scanned / photocopied
    # original whose ink is slightly heavier than the clean digital font.
    try:
        a, _, _ = render_alpha(text, best.path, best.size, ss=1 if best.size < 15 else 4)
        e = _extent(a)
        crop = a[e[1]:e[3], e[0]:e[2]]
        delta = (_stroke_width(obs) - _stroke_width(crop)) / 2.0
        best.weight = float(np.clip(delta, -1.4, 1.8))
    except Exception:
        best.weight = 0.0

    # If the source itself is condensed/expanded (e.g. a stamped label), mirror
    # that width without changing the vertical font size.
    try:
        a, _, _ = render_alpha(text, best.path, best.size, ss=1 if best.size < 15 else 4)
        e = _extent(a)
        rw = max(1, e[2] - e[0])
        best.stretch = float(np.clip(ow / rw, 0.88, 1.12))
    except Exception:
        best.stretch = 1.0
    return best


def estimate_skew(lines: list[list[Word]]) -> float:
    """Median tilt (degrees, image coords: + = text slopes down to the right) of the OCR lines."""
    slopes = []
    for ln in lines:
        if len(ln) >= 3:
            xs = np.array([w.x + w.w / 2 for w in ln])
            ys = np.array([w.y + w.h / 2 for w in ln])
            if xs.max() - xs.min() >= 250:
                slopes.append(np.polyfit(xs, ys, 1)[0])
    if len(slopes) < 2:
        return 0.0
    deg = math.degrees(math.atan(float(np.median(slopes))))
    return deg if 0.3 <= abs(deg) <= 8 else 0.0


@dataclass
class StyleProfile:
    """Every visual attribute the replacement inherits from the source ink.

    Resolving size, colour, weight, width, softness and case in one object
    keeps the renderer and the report in sync: what is measured is exactly
    what gets drawn.
    """
    path: str
    font: str
    size: float
    score: float = 0.0
    weight: float = 0.0          # raw stroke correction measured on the source
    stretch: float = 1.0
    color: tuple = (0.0, 0.0, 0.0)   # BGR, 0..255
    soften: float = 0.45
    case: str = "as-typed"


def _case_kind(text: str) -> str:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return "none"
    if all(c.isupper() for c in letters):
        return "upper"
    if all(c.islower() for c in letters):
        return "lower"
    words = re.findall(r"[^\W\d_]+", text, flags=re.UNICODE)
    if words and all(w[0].isupper() and w[1:].islower() for w in words):
        return "title"
    return "mixed"


def match_source_case(source: str, replacement: str) -> str:
    """Give ``replacement`` the same letter case as the text it replaces.

    Digits and punctuation are untouched.  Deliberately mixed spellings such
    as ``iPhone`` or ``McDonald`` are left exactly as typed, because forcing
    a pattern there would damage a name rather than copy a style.
    """
    kind = _case_kind(source)
    if not replacement or kind in ("none", "mixed"):
        return replacement
    if kind == "upper":
        return replacement.upper()
    if kind == "lower":
        return replacement.lower()
    # Title Case: capitalise every word and lower-case the rest of that word.
    return " ".join(
        p[:1].upper() + p[1:].lower() if p else p
        for p in replacement.split(" ")
    )


def resolve_style(reg: Region, obs: np.ndarray, old_text: str, usable: list[str], *,
                  font_override=None, color=None, size_scale=1.0, soften=None,
                  weight_match=True, stretch_match=True) -> StyleProfile:
    """Resolve font, size, weight, width, colour and softness from the source."""
    if font_override:
        fit = FontFit(font_override, 0.0, 1.0)
        fit.size = _size_for_height(old_text, font_override, obs.shape[0], obs.shape[1])
        try:
            a, _, _ = render_alpha(old_text, font_override, fit.size, ss=1 if fit.size < 15 else 4)
            e = _extent(a)
            crop = a[e[1]:e[3], e[0]:e[2]]
            fit.weight = float(np.clip((_stroke_width(obs) - _stroke_width(crop)) / 2.0, -1.4, 1.8))
            fit.stretch = float(np.clip(obs.shape[1] / max(1, crop.shape[1]), 0.88, 1.12))
        except Exception:
            pass
    else:
        fit = fit_font(usable, old_text, obs)

    path = fit.path or (usable[0] if usable else "")
    size = fit.size or max(4.0, obs.shape[0] * 0.9)
    if size_scale and float(size_scale) != 1.0:
        size = max(4.0, size * float(size_scale))

    ink = color if color is not None else reg.color
    col = tuple(int(round(float(v))) for v in ink)
    soft = float(reg.soften if soften is None else soften)
    return StyleProfile(
        path=path,
        font=font_label(path),
        size=size,
        score=float(fit.score),
        weight=float(fit.weight) if weight_match else 0.0,
        stretch=float(fit.stretch) if stretch_match else 1.0,
        color=col,
        soften=soft,
        case=_case_kind(old_text),
    )


def _prepare_line_alpha(alpha: np.ndarray, sx: float, weight: float, soften: float) -> np.ndarray:
    """Apply line-wide source style without changing individual glyph geometry."""
    a = alpha
    if abs(sx - 1.0) > 0.002:
        a = cv2.resize(a, (max(1, int(round(a.shape[1] * sx))), a.shape[0]),
                       interpolation=cv2.INTER_AREA)
    if abs(weight) >= 0.3:
        a = _weighted_crop(a, float(weight) * 0.55)
    if soften > 0:
        a = cv2.GaussianBlur(a, (0, 0), soften)
    return a


def _source_reconstruction(alpha: np.ndarray, L: float, T: int,
                           origin_x: float, baseline_y: float,
                           obs: np.ndarray, page_x0: int, page_y0: int):
    """Return the calibrated source ink extent and IoU in the level OCR box."""
    oh, ow = obs.shape
    ix, iy = math.floor(origin_x), math.floor(baseline_y)
    origin_fx, origin_fy = origin_x - ix, baseline_y - iy
    margin = 8
    bh, bw = oh + 2 * margin, ow + 2 * margin
    canvas = np.zeros((bh, bw), np.float32)
    px = ix + L - page_x0 + margin
    py = iy + T - page_y0 + margin
    canvas_px, canvas_py = math.floor(px), math.floor(py)
    dx = origin_fx + px - canvas_px
    dy = origin_fy + py - canvas_py
    sx0, sy0 = max(0, -canvas_px), max(0, -canvas_py)
    dx0, dy0 = max(0, canvas_px), max(0, canvas_py)
    sw = min(alpha.shape[1] - sx0, bw - dx0)
    sh = min(alpha.shape[0] - sy0, bh - dy0)
    if sw <= 0 or sh <= 0:
        return None, None
    canvas[dy0:dy0 + sh, dx0:dx0 + sw] = alpha[sy0:sy0 + sh, sx0:sx0 + sw]
    if abs(dx) > 1e-9 or abs(dy) > 1e-9:
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        canvas = cv2.warpAffine(canvas, M, (bw, bh), flags=cv2.INTER_LINEAR)
    reconstructed = canvas[margin:margin + oh, margin:margin + ow] > 0.5
    union = np.logical_or(reconstructed, obs).sum()
    iou = float(np.logical_and(reconstructed, obs).sum() / max(1, union))
    e = _extent(reconstructed.astype(np.float32))
    bbox = None if e is None else [int(page_x0 + e[0]), int(page_y0 + e[1]),
                                   int(page_x0 + e[2]), int(page_y0 + e[3])]
    return bbox, iou


def draw_replacement(out, reg: Region, old_text: str, new_text: str, fonts: list[str], align="left",
                     shrink=False, min_scale=0.6, soften=None, next_x=None, font_override=None, skew=0.0,
                     color=None, size_scale=1.0, preserve_size=True, weight_match=True,
                     stretch_match=True, style_info=None, nudge_x=0.0, nudge_y=0.0, case_match=True,
                     pixel_lock=True):
    cx0, cy0, _, _ = reg.ctx
    core = reg.core
    R = C = None
    if abs(skew) >= 0.3:
        # Measure one level source line; rotate only the finished replacement.
        th = math.radians(skew)
        R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]])
        ys_, xs_ = np.nonzero(core)
        cc = (float(xs_.mean()), float(ys_.mean()))
        Mi = cv2.getRotationMatrix2D(cc, skew, 1.0)
        core = cv2.warpAffine(core.astype(np.uint8), Mi, (core.shape[1], core.shape[0]),
                              flags=cv2.INTER_NEAREST) > 0
        C = np.array([cx0 + cc[0], cy0 + cc[1]])
        ys_, xs_ = np.nonzero(core)
        x0, y0 = cx0 + xs_.min(), cy0 + ys_.min()
        x1, y1 = cx0 + xs_.max() + 1, cy0 + ys_.max() + 1
    else:
        x0, y0, x1, y1 = reg.core_bbox
    obs = core[y0 - cy0:y1 - cy0, x0 - cx0:x1 - cx0]

    applied = match_source_case(old_text, new_text) if case_match else new_text
    usable = [f for f in fonts if has_glyphs(f, old_text + applied)]
    if not usable:
        raise RuntimeError("no available font has all the needed characters; pass --font <file.ttf>")
    profile = resolve_style(
        reg, obs, old_text, usable,
        font_override=font_override,
        color=color,
        size_scale=size_scale,
        soften=soften,
        weight_match=weight_match,
        stretch_match=stretch_match,
    )
    path = profile.path or usable[0]
    size = profile.size or max(4.0, reg.bh * 0.9)
    ss = 1 if size < 15 else 4

    # Calibrate the complete source line once.  The same font rasterisation,
    # line-wide horizontal scale, stroke correction and softness are used for
    # both source reconstruction and replacement.  No per-glyph resizing or
    # slot snapping occurs, and vertical glyph proportions remain natural.
    style_xs = float(profile.stretch) if stretch_match else 1.0
    style_xs = float(np.clip(style_xs, 0.88, 1.12) if pixel_lock else np.clip(style_xs, 0.94, 1.06))
    if abs(style_xs - 1.0) <= 0.002:
        style_xs = 1.0
    applied_weight = float(profile.weight) if weight_match else 0.0
    # Hinted fonts have several raster sizes on the same height/width plateau.
    # A tiny bounded calibration chooses the one that actually reconstructs the
    # source ink; it never scales individual glyphs or changes the user's
    # explicit size_scale override.
    size_offsets = (-0.5, -0.25, 0.0, 0.25, 0.5) if pixel_lock and abs(float(size_scale) - 1.0) < 1e-6 else (0.0,)
    calibrated = None
    for offset in size_offsets:
        candidate_size = max(4.0, size + float(offset))
        candidate_ss = 1 if candidate_size < 15 else 4
        old_a0, old_L, old_T = render_alpha(old_text, path, candidate_size, ss=candidate_ss)
        old_e0 = _extent(old_a0)
        if old_e0 is None:
            continue
        old_a = _prepare_line_alpha(old_a0, style_xs, applied_weight, profile.soften)
        source_ox = x0 - (old_L + old_e0[0]) * style_xs
        source_baseline = y0 - (old_T + old_e0[1])
        cal_dx, cal_dy = _align_to_mask(
            old_a, old_L * style_xs, old_T,
            source_ox, source_baseline, obs, x0, y0,
        )
        source_bbox, source_iou = _source_reconstruction(
            old_a, old_L * style_xs, old_T,
            source_ox + cal_dx, source_baseline + cal_dy, obs, x0, y0,
        )
        if source_iou is None:
            continue
        score = source_iou - abs(float(offset)) * 1e-4
        if calibrated is None or score > calibrated[0]:
            calibrated = (score, candidate_size, candidate_ss, source_ox, source_baseline,
                          cal_dx, cal_dy, source_bbox, source_iou)
    if calibrated is None:
        raise RuntimeError("source text produced no visible glyphs; choose a font with the required characters")
    _, size, ss, source_ox, source_baseline, cal_dx, cal_dy, source_bbox, source_iou = calibrated
    shift_x = cal_dx + float(nudge_x or 0.0)
    shift_y = cal_dy + float(nudge_y or 0.0)

    render_size = size
    xs = style_xs

    def measure(sz):
        alpha, L_, T_ = render_alpha(applied, path, sz, ss=1 if sz < 15 else ss)
        return L_, T_, _extent(alpha)

    new_L, new_T, new_e0 = measure(render_size)
    new_w = 0.0 if new_e0 is None else (new_e0[2] - new_e0[0]) * xs
    old_w = x1 - x0
    note = "whole-line calibrated" if pixel_lock else "whole-line"
    if new_e0 is not None and shrink and not preserve_size and new_w > old_w:
        gap = render_size * 0.3
        if align == "left":
            avail = old_w if next_x is None else max(old_w, next_x - x0 - gap)
            if next_x is None:
                avail = max(old_w, reg.ctx[2] - x0 + reg.bh * 4)
        else:
            avail = old_w
        if new_w > avail:
            need = avail / new_w
            fit_xs = max(need, 0.88)
            us = max(need / fit_xs, min_scale)
            render_size *= us
            xs = style_xs * fit_xs
            new_L, new_T, new_e0 = measure(render_size)
            new_w = 0.0 if new_e0 is None else (new_e0[2] - new_e0[0]) * xs
            note = f"fit {xs:.0%}" + (f", size {us:.0%}" if us < 1 else "")
            if need / fit_xs < min_scale:
                note += " (gap se lamba)"
    elif new_e0 is not None and preserve_size and new_w > old_w * 1.01:
        note = f"source size ({new_w / max(1.0, old_w):.0%} width)"

    # Recover a baseline-relative origin from the source L/T metrics.  Alignment
    # affects the whole line and is bounded by _align_to_mask; tracking remains
    # zero so font kerning and advances are preserved exactly.
    if new_e0 is not None:
        new_rel_l = (new_L + new_e0[0]) * xs
        new_rel_r = (new_L + new_e0[2]) * xs
        if align == "center":
            ox = (x0 + x1) / 2.0 + shift_x - (new_rel_l + new_rel_r) / 2.0
        elif align == "right":
            ox = x1 + shift_x - new_rel_r
        else:
            ox = x0 + shift_x - new_rel_l
    else:
        ox = source_ox + shift_x
    oy = source_baseline + shift_y
    if reg.bh <= 20:
        ox = float(round(ox))
        oy = float(round(oy))
    ix, iy = math.floor(ox), math.floor(oy)
    final_ss = 1 if render_size < 15 else ss
    a, L, T = render_alpha(
        applied, path, render_size,
        fx=(ox - ix) if abs(xs - 1.0) <= 0.002 else 0.0,
        fy=oy - iy,
        ss=final_ss,
    )
    a = _prepare_line_alpha(a, xs, applied_weight, profile.soften)
    actual_xs = xs if abs(xs - 1.0) > 0.002 else 1.0
    if abs(xs - 1.0) > 0.002:
        px = int(round(ox + L * xs))
    else:
        px = ix + L
    py = iy + T
    derotated_e = _extent(a)
    derotated_bbox = None if derotated_e is None else [
        px + derotated_e[0], py + derotated_e[1],
        px + derotated_e[2], py + derotated_e[3],
    ]

    if R is not None:
        # Every replacement, including pixel-lock, uses this common rotation.
        ah, aw = a.shape
        t = R @ np.array([px - C[0], py - C[1]]) + C
        corners = np.array([[0, 0], [aw, 0], [0, ah], [aw, ah]], float) @ R.T + t
        mn = np.floor(corners.min(0)).astype(int) - 1
        mx = np.ceil(corners.max(0)).astype(int) + 1
        M = np.hstack([R, (t - mn).reshape(2, 1)]).astype(np.float32)
        a = cv2.warpAffine(a, M, (int(mx[0] - mn[0]), int(mx[1] - mn[1])), flags=cv2.INTER_LINEAR)
        px, py = int(mn[0]), int(mn[1])

    H, W = out.shape[:2]
    sx0, sy0 = max(0, px), max(0, py)
    sx1, sy1 = min(W, px + a.shape[1]), min(H, py + a.shape[0])
    visible = a[sy0 - py:sy1 - py, sx0 - px:sx1 - px] if sx1 > sx0 and sy1 > sy0 else a[:0, :0]
    visible_e = _extent(visible)
    replacement_bbox = None if visible_e is None else [
        sx0 + visible_e[0], sy0 + visible_e[1],
        sx0 + visible_e[2], sy0 + visible_e[3],
    ]
    pixel_info = {
        "method": "whole-line-calibrated" if pixel_lock else "whole-line",
        "source_ink_bbox": [int(v) for v in reg.ink_bbox],
        "source_core_bbox": [int(v) for v in reg.core_bbox],
        "source_reconstruction_bbox": source_bbox,
        "source_reconstruction_frame": "de-rotated" if R is not None else "page",
        "source_reconstruction_iou": None if source_iou is None else round(float(source_iou), 4),
        "replacement_ink_bbox_derotated": derotated_bbox,
        "replacement_ink_bbox": replacement_bbox,
        "baseline_y_derotated": round(float(oy), 3),
        "baseline_offset": round(float(oy - y1), 3),
        "calibration_shift": [round(float(cal_dx), 3), round(float(cal_dy), 3)],
        "nudge": [round(float(nudge_x or 0.0), 3), round(float(nudge_y or 0.0), 3)],
        "source_scale_x": round(float(style_xs), 4),
        "replacement_scale_x": round(float(actual_xs), 4),
        "replacement_scale_y": 1.0,
        "tracking": 0.0,
        "rotation": round(float(skew if R is not None else 0.0), 3),
    }
    if style_info is not None:
        b, g, r = [int(round(v)) for v in profile.color]
        style_info.update({
            "font": font_label(path),
            "file": os.path.basename(path),
            "size": round(float(render_size), 2),
            "score": round(float(profile.score), 3),
            "weight": round(float(applied_weight * 0.55 if abs(applied_weight) >= 0.3 else 0.0), 2),
            "stretch": round(float(actual_xs), 3),
            "color": "#{:02x}{:02x}{:02x}".format(r, g, b),
            "soften": round(float(profile.soften), 2),
            "shift": [round(float(shift_x), 2), round(float(shift_y), 2)],
            "case": profile.case,
            "applied": applied,
            "pixel": pixel_info,
        })
        if source_iou is not None:
            note += f" · source IoU {source_iou:.3f}"

    if sx1 <= sx0 or sy1 <= sy0:
        return path, note
    al = np.clip(visible, 0, 1)[..., None]
    reg_px = out[sy0:sy1, sx0:sx1].astype(np.float32)
    col = np.array(profile.color, np.float32)
    out[sy0:sy1, sx0:sx1] = np.clip(reg_px * (1 - al) + col * al, 0, 255).astype(np.uint8)

    # If the watermark sits ON TOP of the text, the new glyphs need the same
    # veil as the old ones.  All renderer modes reach this common compositor.
    u = reg.underlay
    if u is not None and u.present and u.order == "over" and u.alpha is not None:
        aa = u.alpha[sy0 - cy0:sy1 - cy0, sx0 - cx0:sx1 - cx0]
        if aa.shape == al.shape[:2] and aa.size:
            veil = np.clip(aa, 0, 1) * np.clip(al[..., 0] * 1.6, 0, 1) * 0.92
            wcol = np.array(u.color, np.float32)
            seg = out[sy0:sy1, sx0:sx1].astype(np.float32)
            out[sy0:sy1, sx0:sx1] = np.clip(
                seg * (1 - veil[..., None]) + wcol * veil[..., None], 0, 255
            ).astype(np.uint8)
    return path, note


# =============================================================================
# High-level edit
# =============================================================================


def edit_page(page: Page, matches: list[Match], new_text: str | None, *, method="auto", align="left",
              shrink=False, soften=None, fonts=None, font_override=None, skew=0.0, log=print,
              color=None, size_scale=1.0, preserve_size=True, weight_match=True,
              stretch_match=True, style_info=None, nudge_x=0.0, nudge_y=0.0,
              case_match=True, underlay_mask=None, watermark_mode="auto",
              watermark_sensitivity=0.5, pixel_lock=True, source_bgr=None) -> np.ndarray:
    src = page.bgr if source_bgr is None else source_bgr
    out = page.bgr.copy()
    fonts = fonts if fonts is not None else (discover_fonts(None) if new_text else [])
    if underlay_mask is None and watermark_mode != "off":
        try:
            underlay_mask = detect_underlay_page(src, watermark_sensitivity)
        except Exception:
            underlay_mask = None
    for m in matches:
        reg = analyze_region(
            src, m.box, underlay_mask=underlay_mask,
            watermark_mode=watermark_mode,
            watermark_sensitivity=watermark_sensitivity,
        )
        if reg is None:
            log(f"  ! skipped '{m.text}': no ink found at that spot (OCR false positive?)")
            continue
        if style_info is not None:
            style_info["watermark"] = _underlay_report(reg.underlay, watermark_mode)
        used = erase(out, src, reg, method)
        msg = f"  - '{m.text}' erased ({used})"
        if reg.underlay is not None and reg.underlay.present:
            msg += (f" [watermark {reg.underlay.order} "
                    f"{int(round(reg.underlay.confidence * 100))}%]")
        if new_text:
            fpath, note = draw_replacement(
                out, reg, m.text, new_text, fonts, align, shrink,
                soften=soften, next_x=m.next_x, font_override=font_override, skew=skew,
                color=color, size_scale=size_scale, preserve_size=preserve_size,
                weight_match=weight_match, stretch_match=stretch_match,
                style_info=style_info, nudge_x=nudge_x, nudge_y=nudge_y,
                case_match=case_match, pixel_lock=pixel_lock,
            )
            msg += f" -> '{new_text}' in {font_label(fpath)} ({os.path.basename(fpath)}) {note}"
        log(msg)
    return out


def preview(page: Page, matches: list[tuple[int, Match]]) -> np.ndarray:
    img = page.bgr.copy()
    for idx, m in matches:
        x0, y0, x1, y1 = [int(v) for v in m.box]
        cv2.rectangle(img, (x0 - 3, y0 - 3), (x1 + 3, y1 + 3), (0, 0, 255), 2)
        cv2.putText(img, str(idx), (x0 - 3, max(14, y0 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return img


# =============================================================================
# CLI
# =============================================================================


def _ask(prompt: str, default: str | None = None) -> str:
    s = input(prompt).strip()
    return s if s or default is None else default


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Find text in a document image/PDF and remove or replace it.")
    ap.add_argument("input")
    ap.add_argument("-o", "--output")
    ap.add_argument("--find", help="text to look for")
    g = ap.add_mutually_exclusive_group()
    g.add_argument("--remove", action="store_true", help="erase the text, keep the background")
    g.add_argument("--replace", metavar="NEW", help="replace the text with NEW")
    ap.add_argument("--all", action="store_true", help="apply to every match without asking")
    ap.add_argument("--only", help="comma-separated match numbers to apply, e.g. 1,3")
    ap.add_argument("--lang", default="eng", help="tesseract language(s), e.g. eng+hin")
    ap.add_argument("--psm", type=int, default=3, help="tesseract page segmentation mode (try 6 or 11)")
    ap.add_argument("--case-sensitive", action="store_true")
    ap.add_argument("--whole-word", action="store_true")
    ap.add_argument("--fuzzy", type=float, default=0.8, help="OCR-error tolerance when no exact hit (0 = off)")
    ap.add_argument("--method", choices=["auto", "fill", "inpaint"], default="auto")
    ap.add_argument("--font", action="append", help="font file/dir to use for replacement (repeatable)")
    ap.add_argument("--align", choices=["left", "center", "right"], default="left")
    ap.add_argument("--no-shrink", action="store_true", help="never shrink longer replacement text")
    ap.add_argument("--soften", type=float, default=0.5, help="blur on new text edges (match scan softness)")
    ap.add_argument("--dpi", type=int, default=300, help="rasterising DPI for PDFs")
    ap.add_argument("--enhance", action="store_true", help="always pre-process the page for OCR")
    ap.add_argument("--dump-ocr", action="store_true", help="print what OCR reads, then exit")
    a = ap.parse_args(argv)

    pages = load_pages(a.input, a.dpi)
    print(f"Loaded {len(pages)} page(s). Running OCR ...")
    all_lines = [ocr_lines(p.bgr, a.lang, a.psm, enhance=a.enhance) for p in pages]

    if a.dump_ocr:
        for i, lines in enumerate(all_lines):
            print(f"--- page {i + 1}")
            for ln in lines:
                print(" ".join(w.text for w in ln), f"  (conf {np.mean([w.conf for w in ln]):.0f})")
        return 0

    query = a.find or _ask("Text to find: ")
    matches = find_matches(all_lines, query, a.case_sensitive, a.whole_word, a.fuzzy)
    if not matches:
        print("No match - retrying OCR with image enhancement (textured/uneven backgrounds) ...")
        all_lines = [ocr_lines(p.bgr, a.lang, a.psm, enhance=True) for p in pages]
        matches = find_matches(all_lines, query, a.case_sensitive, a.whole_word, a.fuzzy)
    if not matches:
        print("No match. Run with --dump-ocr to see what the OCR reads; try --psm 6/11 or another --lang.")
        return 1

    print(f"\n{len(matches)} match(es):")
    for i, m in enumerate(matches, 1):
        tag = "" if m.score >= 1 else f"  (fuzzy {m.score:.0%})"
        print(f"  [{i}] page {m.page + 1}: ...{m.context}...{tag}")
    stem = Path(a.input).with_suffix("")
    for pno, p in enumerate(pages):
        pm = [(i, m) for i, m in enumerate(matches, 1) if m.page == pno]
        if pm:
            pv = f"{stem}_preview{'' if len(pages) == 1 else f'_p{pno + 1}'}.png"
            cv2.imwrite(pv, preview(p, pm))
            print(f"  preview with numbered boxes -> {pv}")

    new_text = a.replace
    if not a.remove and a.replace is None:
        act = _ask("\nAction  [r] remove   [p] replace : ").lower()
        if act.startswith("p"):
            new_text = _ask("Replacement text: ")
        else:
            new_text = None
    chosen = list(range(1, len(matches) + 1))
    if a.only:
        chosen = [int(x) for x in a.only.split(",") if x.strip().isdigit()]
    elif len(matches) > 1 and not a.all and sys.stdin.isatty():
        s = _ask("Apply to which matches? [Enter = all, or e.g. 1,3] : ")
        if s:
            chosen = [int(x) for x in s.split(",") if x.strip().isdigit()]

    print("\nEditing ...")
    fonts = discover_fonts(a.font) if new_text else None
    results = []
    for pno, p in enumerate(pages):
        pm = [matches[i - 1] for i in chosen if 1 <= i <= len(matches) and matches[i - 1].page == pno]
        if not pm:
            results.append(p)
            continue
        skew = estimate_skew(all_lines[pno]) if new_text else 0.0
        if skew:
            print(f"  page {pno + 1}: detected text tilt {skew:+.2f} deg - new text will follow it")
        new_bgr = edit_page(p, pm, new_text, skew=skew, method=a.method, align=a.align, shrink=not a.no_shrink,
                            soften=a.soften, fonts=fonts, font_override=(a.font[0] if a.font and len(a.font) == 1 and os.path.isfile(os.path.expanduser(a.font[0])) else None))
        changed = np.any(new_bgr != p.bgr, axis=2).mean() * 100
        print(f"  page {pno + 1}: {changed:.3f}% of pixels changed, the rest is bit-identical")
        results.append(Page(new_bgr, p.alpha, p.meta))

    out = a.output or f"{stem}_edited{Path(a.input).suffix}"
    save_pages(results, out)
    print(f"\nSaved -> {out}")
    if Path(out).suffix.lower() in (".jpg", ".jpeg"):
        print("Note: JPEG output is re-encoded (q97, 4:4:4) so every pixel shifts very slightly. "
              "Use a .png output for a truly lossless result.")
    return 0


#if __name__ == "__main__":
#    sys.exit(main())


# =============================================================================
# TextSwap Studio - UI backend
# =============================================================================
import base64
import json
import mimetypes
import os
import re
import shutil
import subprocess
import tempfile
import time
import traceback
import uuid
import zipfile
from pathlib import Path

_APP_ROOT = Path(tempfile.gettempdir()) / "textswap_studio"
_APP_ROOT.mkdir(parents=True, exist_ok=True)

_SESSIONS = {}
_SESSION_TTL = 3 * 3600
_MAX_FILES = 12
_MAX_UPLOAD = 30 * 1024 * 1024
_MAX_PAIRS = 10
_MAX_PREVIEW_PAGES = 5
_MAX_DOWNLOAD = 55 * 1024 * 1024
_MAX_ZIP = 70 * 1024 * 1024
_TEXT_PAYLOAD_LIMIT = 220_000
_MATCH_PAYLOAD_LIMIT = 250

_ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".jfif", ".webp", ".bmp", ".tif", ".tiff", ".gif", ".pdf"}
_ALLOWED_FORMATS = {"png", "jpg", "webp", "pdf"}
_ALLOWED_METHODS = {"auto", "fill", "inpaint"}
_ALLOWED_ALIGNS = {"left", "center", "right"}
_ALLOWED_PSM = {3, 4, 6, 8, 10, 11, 12, 13}

_LANG_OPTIONS = [
    {"code": "eng", "label": "English", "flag": "🇬🇧"},
    {"code": "hin", "label": "हिन्दी", "flag": "🇮🇳"},
    {"code": "urd", "label": "اردو", "flag": "🇵🇰"},
    {"code": "ara", "label": "العربية", "flag": "🇸🇦"},
    {"code": "ben", "label": "বাংলা", "flag": "🇧🇩"},
    {"code": "tam", "label": "தமிழ்", "flag": "🇮🇳"},
    {"code": "tel", "label": "తెలుగు", "flag": "🇮🇳"},
    {"code": "mar", "label": "मराठी", "flag": "🇮🇳"},
    {"code": "guj", "label": "ગુજરાતી", "flag": "🇮🇳"},
    {"code": "pan", "label": "ਪੰਜਾਬੀ", "flag": "🇮🇳"},
    {"code": "san", "label": "संस्कृतम्", "flag": "🇮🇳"},
    {"code": "fra", "label": "Français", "flag": "🇫🇷"},
    {"code": "deu", "label": "Deutsch", "flag": "🇩🇪"},
    {"code": "spa", "label": "Español", "flag": "🇪🇸"},
    {"code": "por", "label": "Português", "flag": "🇵🇹"},
    {"code": "ita", "label": "Italiano", "flag": "🇮🇹"},
    {"code": "nld", "label": "Nederlands", "flag": "🇳🇱"},
    {"code": "rus", "label": "Русский", "flag": "🇷🇺"},
    {"code": "tur", "label": "Türkçe", "flag": "🇹🇷"},
    {"code": "chi_sim", "label": "中文 (简体)", "flag": "🇨🇳"},
    {"code": "chi_tra", "label": "中文 (繁體)", "flag": "🇹🇼"},
    {"code": "jpn", "label": "日本語", "flag": "🇯🇵"},
    {"code": "kor", "label": "한국어", "flag": "🇰🇷"},
]
_ALLOWED_LANGS = {x["code"] for x in _LANG_OPTIONS}
_APT_LANG = {c: "tesseract-ocr-" + c.replace("_", "-") for c in _ALLOWED_LANGS}


def _cleanup_sessions():
    now = time.time()
    for tok, sess in list(_SESSIONS.items()):
        if now - sess.get("created", now) > _SESSION_TTL:
            shutil.rmtree(sess.get("dir", ""), ignore_errors=True)
            _SESSIONS.pop(tok, None)
    if len(_SESSIONS) > 20:
        oldest = sorted(_SESSIONS.items(), key=lambda kv: kv[1].get("created", 0))
        for tok, sess in oldest[: len(_SESSIONS) - 20]:
            shutil.rmtree(sess.get("dir", ""), ignore_errors=True)
            _SESSIONS.pop(tok, None)


def _new_session(token):
    _cleanup_sessions()
    root = Path(tempfile.mkdtemp(prefix="run_", dir=str(_APP_ROOT)))
    sess = {
        "dir": root,
        "created": time.time(),
        "files": [],
        "results": [],
        "combined_id": None,
    }
    _SESSIONS[token] = sess
    return sess


def _get_session(token):
    if not isinstance(token, str) or not re.fullmatch(r"[0-9a-f]{32}", token):
        raise ValueError("Session invalid hai. Notebook cell dobara run karein.")
    sess = _SESSIONS.get(token)
    if sess is None:
        sess = _new_session(token)
    return sess


def _safe_name(name, fallback="upload"):
    name = Path(str(name or "")).name
    name = re.sub(r"[^\w.\- ()\[\]]+", "_", name, flags=re.UNICODE)
    name = name.strip(" .")[:140]
    return name or fallback


def _safe_stem(name, fallback="file"):
    stem = Path(str(name or "")).stem
    stem = re.sub(r"[^\w.\-]+", "_", stem, flags=re.UNICODE).strip(" ._")
    return (stem or fallback)[:80]


_FONT_CHOICES = None


def _font_choices():
    """Installed fonts exposed as an optional manual override in Advanced."""
    global _FONT_CHOICES
    if _FONT_CHOICES is None:
        try:
            paths = discover_fonts(None)
        except Exception:
            paths = []
        seen = set()
        out = []
        for path in paths:
            label = font_label(path)
            key = label.lower()
            if key in seen:
                continue
            seen.add(key)
            out.append({"label": label, "path": str(path)})
        out.sort(key=lambda x: x["label"].lower())
        _FONT_CHOICES = out[:90]
    return _FONT_CHOICES


def _hex_to_bgr(value):
    if not value:
        return None
    s = str(value).strip().lstrip("#")
    if len(s) == 3:
        s = "".join(c * 2 for c in s)
    if not re.fullmatch(r"[0-9a-fA-F]{6}", s):
        return None
    r, g, b = int(s[0:2], 16), int(s[2:4], 16), int(s[4:6], 16)
    return (b, g, r)


def _run_shell(cmd, timeout=900):
    return subprocess.run(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout
    )


def _available_langs():
    try:
        return set(pytesseract.get_languages(config=""))
    except Exception:
        return {"eng"}


def _ensure_langs(codes, notes):
    codes = [c for c in (codes or []) if c in _ALLOWED_LANGS]
    if not codes:
        codes = ["eng"]
    # keep order, remove duplicates
    codes = list(dict.fromkeys(codes))
    avail = _available_langs()
    missing = [c for c in codes if c not in avail]
    if missing and os.name != "nt" and shutil.which("apt-get"):
        pkgs = " ".join(_APT_LANG[c] for c in missing)
        cmd = (
            f"apt-get install -y -qq --no-install-recommends {pkgs} "
            f"|| (apt-get update -qq && apt-get install -y -qq --no-install-recommends {pkgs})"
        )
        try:
            r = _run_shell(cmd, timeout=900)
            if r.returncode != 0:
                notes.append(
                    "Language pack install nahi hua; available language ke saath try kiya."
                )
        except Exception:
            notes.append("Language pack install check fail hua; available language use ho rahi hai.")
        avail = _available_langs()
    usable = [c for c in codes if c in avail]
    if not usable:
        usable = ["eng"] if "eng" in avail else (sorted(avail)[:1] or ["eng"])
        notes.append("Selected OCR language available nahi thi; English fallback use hua.")
    elif len(usable) < len(codes):
        skipped = ", ".join(c for c in codes if c not in avail)
        notes.append(f"Ye language pack available nahi: {skipped}.")
    return "+".join(usable), usable


def _preprocess_for_ocr(bgr, opts):
    img = bgr.copy()
    if opts.get("auto_contrast"):
        try:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            l = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8)).apply(l)
            img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
        except Exception:
            pass
    if opts.get("denoise"):
        try:
            img = cv2.fastNlMeansDenoisingColored(img, None, 4, 4, 7, 21)
        except Exception:
            pass
    if opts.get("grayscale"):
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    if opts.get("binarize"):
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if opts.get("binarize_mode") == "adaptive":
            g = cv2.adaptiveThreshold(
                g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
            )
        else:
            _, g = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        img = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    if opts.get("invert"):
        img = cv2.bitwise_not(img)
    return img


def _run_ocr(pages, settings, notes):
    lang, used = _ensure_langs(settings["langs"], notes)
    enhance = settings["enhance"]
    psm = settings["psm"]
    min_conf = settings["min_conf"]
    all_lines = []
    for page in pages:
        proc = _preprocess_for_ocr(page.bgr, settings["preprocess"])
        try:
            lines = ocr_lines(proc, lang=lang, psm=psm, enhance=enhance)
        except Exception:
            if len(used) > 1:
                notes.append("Multi-language OCR fail hua; pehli language se retry kiya.")
                lines = ocr_lines(proc, lang=used[0], psm=psm, enhance=enhance)
            else:
                raise
        if min_conf > 0:
            lines = [[w for w in ln if float(w.conf) >= min_conf] for ln in lines]
            lines = [ln for ln in lines if ln]
        all_lines.append(lines)
    return all_lines, lang


def _join_lines(lines):
    out = []
    for ln in lines:
        t = " ".join(w.text for w in ln).strip()
        if t:
            out.append(t)
    return "\n".join(out)


def _page_texts(all_lines):
    texts = [_join_lines(lines) for lines in all_lines]
    if len(texts) == 1:
        return texts[0]
    parts = []
    for i, t in enumerate(texts, 1):
        parts.append(f"--- Page {i} ---\n{t}".rstrip())
    return "\n\n".join(parts)


def _ocr_confidence(lines):
    vals = [float(w.conf) for ln in lines for w in ln if 0 <= float(w.conf) <= 100]
    return round(sum(vals) / len(vals), 1) if vals else 0.0


def _box_overlap(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return not (ax1 <= bx0 or bx1 <= ax0 or ay1 <= by0 or by1 <= ay0)


def _find_selected(all_lines, pairs, settings, notes):
    selected = []
    used = {}
    counts = {}
    for pi, (find, repl) in enumerate(pairs):
        for pno, lines in enumerate(all_lines):
            try:
                matches = find_matches(
                    [lines],
                    find,
                    case_sensitive=settings["case_sensitive"],
                    whole_word=settings["whole_word"],
                    fuzzy=settings["fuzzy"],
                )
            except Exception:
                matches = []
            matches.sort(key=lambda m: (-m.score, m.box[1], m.box[0]))
            page_used = used.setdefault(pno, [])
            for m in matches:
                m.page = pno
                if settings["first_only"] and counts.get((pi, pno), 0) >= 1:
                    continue
                if any(_box_overlap(m.box, old) for old in page_used):
                    notes.append(
                        f"'{find}' ka ek match pehle wale replacement se overlap tha, isliye skip hua."
                    )
                    continue
                page_used.append(m.box)
                counts[(pi, pno)] = counts.get((pi, pno), 0) + 1
                selected.append({"match": m, "pair": pi, "find": find, "replace": repl})
    selected.sort(key=lambda x: (x["match"].page, x["match"].box[1], x["match"].box[0], x["pair"]))
    return selected


def _apply_edits(pages, all_lines, selected, settings, notes):
    fonts = []
    if any(item["replace"] != "" for item in selected):
        try:
            fonts = discover_fonts(None)
        except Exception:
            fonts = []
        if not fonts:
            notes.append("System font nahi mila; replacement text render nahi ho payega.")
    color = _hex_to_bgr(settings.get("color"))
    edited = []
    logs = []
    styles = []
    for pno, page in enumerate(pages):
        cur = page
        page_items = [x for x in selected if x["match"].page == pno]
        if not page_items:
            edited.append(page)
            continue
        wants_text = any(x["replace"] != "" for x in page_items)
        try:
            skew = estimate_skew(all_lines[pno]) if (wants_text and settings["auto_skew"]) else 0.0
        except Exception:
            skew = 0.0
        # A watermark mask belongs to the page, not to a single match.  Build
        # it once and reuse it for every pair on this page; rebuilding per
        # match is both slow and can pick up already-edited pixels.
        page_underlay = None
        if settings["watermark_mode"] != "off":
            try:
                page_underlay = detect_underlay_page(
                    pages[pno].bgr, settings["watermark_sensitivity"]
                )
            except Exception:
                page_underlay = None
        for item in page_items:
            new_text = item["replace"] if item["replace"] != "" else None

            def _log(msg, _p=pno):
                logs.append(f"page {_p + 1}: {msg}")

            style_info = {}
            try:
                new_bgr = edit_page(
                    cur,
                    [item["match"]],
                    new_text,
                    method=settings["method"],
                    align=settings["align"],
                    shrink=settings["shrink"],
                    soften=None if settings["soften_auto"] else settings["soften"],
                    fonts=fonts,
                    font_override=settings["font_override"],
                    color=color,
                    size_scale=settings["size_scale"],
                    preserve_size=settings["preserve_size"],
                    case_match=settings["case_match"],
                    weight_match=settings["weight_match"],
                    stretch_match=settings["stretch_match"],
                    nudge_x=settings["nudge_x"],
                    nudge_y=settings["nudge_y"],
                    skew=skew,
                    log=_log,
                    style_info=style_info,
                    underlay_mask=page_underlay,
                    watermark_mode=settings["watermark_mode"],
                    watermark_sensitivity=settings["watermark_sensitivity"],
                    pixel_lock=settings["pixel_lock"],
                    source_bgr=pages[pno].bgr,
                )
                cur = Page(new_bgr, page.alpha, page.meta)
                if style_info:
                    style_info = dict(style_info)
                    style_info["page"] = pno + 1
                    style_info["find"] = item["find"]
                    style_info["replace"] = item["replace"]
                    item["style"] = style_info
                    styles.append(style_info)
            except Exception as exc:
                notes.append(f"'{item['find']}' replace nahi hua: {exc}")
        edited.append(cur)
    return edited, logs, styles


def _draw_match_boxes(bgr, matches):
    img = bgr.copy()
    for idx, item in enumerate(matches, 1):
        m = item["match"] if isinstance(item, dict) else item
        x0, y0, x1, y1 = [int(round(v)) for v in m.box]
        cv2.rectangle(img, (max(0, x0 - 3), max(0, y0 - 3)), (x1 + 3, y1 + 3), (52, 52, 225), 2)
        label = str(idx)
        ty = y0 - 8 if y0 > 18 else y1 + 18
        cv2.putText(img, label, (max(2, x0), max(14, ty)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (52, 52, 225), 2)
    return img


def _draw_line_boxes(bgr, lines):
    img = bgr.copy()
    for ln in lines:
        if not ln:
            continue
        x0 = min(w.x for w in ln)
        y0 = min(w.y for w in ln)
        x1 = max(w.x + w.w for w in ln)
        y1 = max(w.y + w.h for w in ln)
        cv2.rectangle(img, (max(0, x0 - 2), max(0, y0 - 2)), (x1 + 2, y1 + 2), (72, 168, 72), 1)
    return img


def _preview_b64(bgr, quality=72, max_w=960):
    h, w = bgr.shape[:2]
    if w > max_w:
        scale = max_w / float(w)
        bgr = cv2.resize(bgr, (max_w, max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        return ""
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


def _write_text(path, text):
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        fh.write(text)


def _register_file(sess, path, label, mime=None):
    path = Path(path)
    rec = {
        "id": uuid.uuid4().hex[:12],
        "path": str(path),
        "filename": path.name,
        "label": label,
        "size": path.stat().st_size,
        "mime": mime or mimetypes.guess_type(path.name)[0] or "application/octet-stream",
    }
    sess["files"].append(rec)
    return {
        "id": rec["id"],
        "label": label,
        "filename": rec["filename"],
        "size": rec["size"],
        "mime": rec["mime"],
    }


def _find_file(sess, fid):
    return next((f for f in sess["files"] if f["id"] == fid), None)


def _write_combined(sess):
    rows = [r for r in sess["results"] if r.get("text")]
    if not rows:
        return None
    parts = []
    for r in rows:
        parts.append(f"===== {r['filename']} =====\n{r.get('_text_full') or r['text']}")
    content = "\n\n".join(parts).strip() + "\n"
    path = Path(sess["dir"]) / "all_ocr_text.txt"
    _write_text(path, content)
    sess["files"] = [
        f for f in sess["files"] if f.get("label") != "Combined OCR text (.txt)"
    ]
    sess["combined_id"] = None
    dl = _register_file(sess, path, "Combined OCR text (.txt)", "text/plain")
    sess["combined_id"] = dl["id"]
    return dl


def _save_outputs(sess, rid, stem, pages, settings):
    fmt = settings["output_format"]
    quality = settings["quality"]
    outdir = Path(sess["dir"]) / rid
    outdir.mkdir(parents=True, exist_ok=True)
    downloads = []
    if fmt == "pdf":
        path = outdir / f"{stem}_textswap.pdf"
        save_pages(pages, str(path))
        downloads.append(_register_file(sess, path, "Edited PDF", "application/pdf"))
        return downloads, [path]
    ext = {"png": ".png", "jpg": ".jpg", "webp": ".webp"}[fmt]
    multi = len(pages) > 1
    paths = []
    for i, page in enumerate(pages):
        name = f"{stem}_textswap" + (f"_p{i + 1}" if multi else "") + ext
        path = outdir / name
        pil = _to_pil(page)
        if fmt == "jpg":
            if pil.mode in ("RGBA", "LA"):
                base = Image.new("RGB", pil.size, (255, 255, 255))
                base.paste(pil, mask=pil.split()[-1])
                pil = base
            else:
                pil = pil.convert("RGB")
            pil.save(path, "JPEG", quality=quality, subsampling=0)
        elif fmt == "webp":
            pil.save(path, "WEBP", quality=quality, method=6)
        else:
            pil.save(path, "PNG", optimize=True)
        label = "Edited image" if len(pages) == 1 else f"Edited page {i + 1}"
        downloads.append(_register_file(sess, path, label))
        paths.append(path)
    return downloads, paths


def _save_ocr_overlays(sess, rid, stem, pages, overlays):
    outdir = Path(sess["dir"]) / rid
    outdir.mkdir(parents=True, exist_ok=True)
    downloads = []
    paths = []
    multi = len(pages) > 1
    for i, (page, overlay) in enumerate(zip(pages, overlays)):
        name = f"{stem}_ocr_overlay" + (f"_p{i + 1}" if multi else "") + ".png"
        path = outdir / name
        pil = _to_pil(Page(overlay, page.alpha, page.meta))
        pil.save(path, "PNG", optimize=True)
        label = "OCR overlay" if len(pages) == 1 else f"OCR overlay page {i + 1}"
        downloads.append(_register_file(sess, path, label))
        paths.append(path)
    return downloads, paths


def _validate_settings(raw):
    raw = raw if isinstance(raw, dict) else {}

    def _bool(key, default=False):
        v = raw.get(key, default)
        return v if isinstance(v, bool) else default

    def _int(key, default, lo, hi):
        try:
            v = int(raw.get(key, default))
        except Exception:
            v = default
        return max(lo, min(hi, v))

    def _float(key, default, lo, hi):
        try:
            v = float(raw.get(key, default))
        except Exception:
            v = default
        return max(lo, min(hi, v))

    pre_raw = raw.get("preprocess") if isinstance(raw.get("preprocess"), dict) else {}
    pre = {
        "auto_contrast": bool(pre_raw.get("auto_contrast", False)),
        "denoise": bool(pre_raw.get("denoise", False)),
        "grayscale": bool(pre_raw.get("grayscale", False)),
        "binarize": bool(pre_raw.get("binarize", False)),
        "invert": bool(pre_raw.get("invert", False)),
        "binarize_mode": "adaptive" if pre_raw.get("binarize_mode") == "adaptive" else "otsu",
    }

    pairs_raw = raw.get("pairs") if isinstance(raw.get("pairs"), list) else []
    pairs = []
    for p in pairs_raw[:_MAX_PAIRS]:
        if not isinstance(p, dict):
            continue
        find = str(p.get("find") or "").strip()[:400]
        if not find:
            continue
        replace = str(p.get("replace") or "")[:400]
        pairs.append((find, replace))

    mode = "ocr" if raw.get("mode") == "ocr" else "swap"
    if mode == "swap" and not pairs:
        raise ValueError("Kam se kam ek 'Find' text likhein (ya 'Sirf OCR' mode chunein).")

    langs = raw.get("langs") if isinstance(raw.get("langs"), list) else []
    langs = [str(x) for x in langs if str(x) in _ALLOWED_LANGS][:6] or ["eng"]

    fmt = str(raw.get("output_format", "png")).lower()
    if fmt not in _ALLOWED_FORMATS:
        fmt = "png"
    method = str(raw.get("method", "auto")).lower()
    if method not in _ALLOWED_METHODS:
        method = "auto"
    align = str(raw.get("align", "left")).lower()
    if align not in _ALLOWED_ALIGNS:
        align = "left"
    watermark_mode = str(raw.get("watermark_mode", "auto")).lower()
    if watermark_mode not in {"auto", "under", "over", "off"}:
        watermark_mode = "auto"
    try:
        psm = int(raw.get("psm", 3))
    except Exception:
        psm = 3
    if psm not in _ALLOWED_PSM:
        psm = 3
    color = raw.get("color")
    if color is not None and _hex_to_bgr(color) is None:
        color = None
    name_prefix = str(raw.get("name_prefix") or "")[:80]
    font_override = str(raw.get("font_override") or "")
    if font_override not in {x["path"] for x in _font_choices()}:
        font_override = None

    return {
        "mode": mode,
        "langs": langs,
        "psm": psm,
        "enhance": _bool("enhance", True),
        "preprocess": pre,
        "pairs": pairs,
        "fuzzy": _float("fuzzy", 0.8, 0.0, 1.0),
        "case_sensitive": _bool("case_sensitive", False),
        "whole_word": _bool("whole_word", False),
        "first_only": _bool("first_only", False),
        "method": method,
        "align": align,
        "shrink": _bool("shrink", False),
        "preserve_size": _bool("preserve_size", True),
        "case_match": _bool("case_match", True),
        "pixel_lock": _bool("pixel_lock", True),
        "watermark_mode": watermark_mode,
        "watermark_sensitivity": _float("watermark_sensitivity", 0.5, 0.0, 1.0),
        "weight_match": _bool("weight_match", True),
        "stretch_match": _bool("stretch_match", True),
        "nudge_x": _float("nudge_x", 0.0, -20.0, 20.0),
        "nudge_y": _float("nudge_y", 0.0, -20.0, 20.0),
        "soften": _float("soften", 0.5, 0.0, 2.0),
        "soften_auto": _bool("soften_auto", True),
        "font_override": font_override,
        "auto_skew": _bool("auto_skew", True),
        "color": color,
        "size_scale": _float("size_scale", 1.0, 0.5, 2.0),
        "output_format": fmt,
        "quality": _int("quality", 92, 60, 100),
        "name_prefix": name_prefix,
        "dpi": _int("dpi", 300, 120, 600),
        "max_pages": _int("max_pages", 0, 0, 50),
        "box_preview": _bool("box_preview", True),
        "min_conf": _int("min_conf", 0, 0, 100),
    }


def _decode_upload(req):
    b64 = req.get("b64") or ""
    if not isinstance(b64, str) or not b64:
        raise ValueError("File data missing hai.")
    if b64.lstrip().lower().startswith("data:"):
        b64 = b64.split(",", 1)[-1]
    try:
        raw = base64.b64decode(b64, validate=False)
    except Exception:
        raise ValueError("File data decode nahi hua. Dobara upload karein.")
    if not raw:
        raise ValueError("File khali hai.")
    if len(raw) > _MAX_UPLOAD:
        raise ValueError(f"File {_MAX_UPLOAD // (1024 * 1024)} MB se badi hai.")
    return raw


def _dedupe(notes):
    out = []
    for n in notes:
        n = str(n).strip()
        if n and n not in out:
            out.append(n)
    return out[:30]


def _friendly_error(exc):
    name = type(exc).__name__
    text = str(exc)
    if name == "TesseractNotFoundError":
        return "Tesseract OCR install nahi mila. Notebook cell dobara run karein (setup automatic hai)."
    if name == "TesseractError":
        return f"OCR engine error: {text}. Language pack ya PSM check karein."
    if name == "MemoryError":
        return "Image bahut badi hai. Chhoti image ya kam DPI se try karein."
    if name in ("UnidentifiedImageError", "DecompressionBombError"):
        return "File valid image nahi hai ya corrupt hai. Dobara upload karein."
    if name in ("FileDataError", "EmptyFileError"):
        return "PDF read nahi ho payi. File corrupt, empty ya password-protected ho sakti hai."
    if isinstance(exc, ValueError):
        return text
    return f"{name}: {text}"


def _cb_process(token, payload):
    try:
        req = payload if isinstance(payload, dict) else json.loads(payload)
        if not isinstance(req, dict):
            raise ValueError("Request format invalid hai.")
        sess = _get_session(token)
        settings = _validate_settings(req.get("settings") or {})
        filename = _safe_name(req.get("filename"), "image")
        ext = Path(filename).suffix.lower()
        if ext not in _ALLOWED_EXTS:
            raise ValueError("Sirf image (PNG/JPG/WEBP/TIFF) ya PDF upload karein.")
        raw = _decode_upload(req)
        if ext == ".pdf":
            try:
                import pymupdf  # noqa: F401
            except Exception:
                try:
                    import fitz  # noqa: F401
                except Exception:
                    raise ValueError("PDF support ke liye pymupdf install nahi hua. Cell dobara run karein.")

        t0 = time.time()
        rid = uuid.uuid4().hex[:10]
        work = Path(sess["dir"]) / rid
        work.mkdir(parents=True, exist_ok=True)
        inpath = work / ("input" + ext)
        inpath.write_bytes(raw)

        notes = []
        pages = load_pages(str(inpath), dpi=settings["dpi"])
        if not pages:
            raise ValueError("File me koi readable page nahi mila.")
        if settings["max_pages"] and len(pages) > settings["max_pages"]:
            notes.append(
                f"PDF ke sirf pehle {settings['max_pages']} page process kiye (total {len(pages)})."
            )
            pages = pages[: settings["max_pages"]]

        all_lines, lang_used = _run_ocr(pages, settings, notes)
        text = _page_texts(all_lines)
        word_count = sum(len(ln) for ln in all_lines)
        confidence = round(
            sum(_ocr_confidence(lines) for lines in all_lines) / max(1, len(all_lines)), 1
        )

        mode = settings["mode"]
        selected = []
        edited = list(pages)
        logs = []
        styles = []
        status = "ocr"
        if mode == "swap":
            selected = _find_selected(all_lines, settings["pairs"], settings, notes)
            if selected:
                edited, logs, styles = _apply_edits(pages, all_lines, selected, settings, notes)
                status = "done"
            else:
                status = "no_match"
                notes.append(
                    "Find text OCR me nahi mila. Spelling, language, PSM ya fuzzy slider check karein."
                )
        elif not text.strip():
            status = "no_text"
            notes.append("Is image me OCR ko koi text nahi mila. Preprocessing ya PSM badal kar dekhein.")

        stem = _safe_stem(filename, "file")
        if settings["name_prefix"]:
            stem = _safe_stem(settings["name_prefix"].replace("{name}", stem), stem)

        downloads = []
        if mode == "ocr":
            overlays = []
            for i, lines in enumerate(all_lines):
                overlays.append(_draw_line_boxes(pages[i].bgr, lines) if settings["box_preview"] else pages[i].bgr)
            dl_img, _ = _save_ocr_overlays(sess, rid, stem, pages, overlays)
            downloads.extend(dl_img)
        else:
            dl_img, _ = _save_outputs(sess, rid, stem, edited, settings)
            downloads.extend(dl_img)

        text_path = work / f"{stem}_ocr.txt"
        _write_text(text_path, text)
        downloads.append(_register_file(sess, text_path, "OCR text (.txt)", "text/plain"))

        report = {
            "app": "TextSwap Studio",
            "version": "2.0",
            "filename": filename,
            "processed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "mode": mode,
            "status": status,
            "pages": len(pages),
            "ocr_language": lang_used,
            "ocr_confidence": confidence,
            "word_count": word_count,
            "settings": settings,
            "text": text,
            "matches": [
                {
                    "pair": item["pair"] + 1,
                    "find": item["find"],
                    "replace": item["replace"],
                    "matched_text": item["match"].text,
                    "page": item["match"].page + 1,
                    "box": [round(float(v), 1) for v in item["match"].box],
                    "score": round(float(item["match"].score), 4),
                    "context": item["match"].context,
                    "style": item.get("style"),
                }
                for item in selected[:2000]
            ],
            "styles": styles[:2000],
            "warnings": _dedupe(notes),
            "logs": logs[:500],
        }
        report_path = work / f"{stem}_report.json"
        _write_text(report_path, json.dumps(report, ensure_ascii=False, indent=2))
        downloads.append(_register_file(sess, report_path, "Report (.json)", "application/json"))

        previews = []
        for i in range(min(len(pages), _MAX_PREVIEW_PAGES)):
            if mode == "swap" and selected:
                page_matches = [x for x in selected if x["match"].page == i]
                before = _draw_match_boxes(pages[i].bgr, page_matches) if settings["box_preview"] else pages[i].bgr
            elif mode == "ocr":
                before = _draw_line_boxes(pages[i].bgr, all_lines[i]) if settings["box_preview"] else pages[i].bgr
            else:
                before = pages[i].bgr
            after = edited[i].bgr if mode == "swap" else pages[i].bgr
            previews.append(
                {
                    "page": i + 1,
                    "before": _preview_b64(before),
                    "after": _preview_b64(after),
                }
            )
        if len(pages) > _MAX_PREVIEW_PAGES:
            notes.append(
                f"Preview sirf pehle {_MAX_PREVIEW_PAGES} page ka hai; baaki pages download me hain."
            )

        changed = False
        if mode == "swap" and selected:
            changed = any(
                np.any(edited[i].bgr != pages[i].bgr) for i in range(len(pages))
            )
            if not changed:
                status = "no_change"
                notes.append(
                    "Match mila par image me visible change nahi hua. Font / erase settings check karein."
                )

        match_payload = [
            {
                "pair": item["pair"] + 1,
                "find": item["find"],
                "replace": item["replace"],
                "matched_text": item["match"].text,
                "page": item["match"].page + 1,
                "score": round(float(item["match"].score), 3),
                "context": item["match"].context[:600],
                "box": [round(float(v), 1) for v in item["match"].box],
                "style": item.get("style"),
            }
            for item in selected[:_MATCH_PAYLOAD_LIMIT]
        ]
        if len(selected) > _MATCH_PAYLOAD_LIMIT:
            notes.append(
                f"UI me pehle {_MATCH_PAYLOAD_LIMIT} matches dikh rahe hain; poora report download me hai."
            )
        if len(text) > _TEXT_PAYLOAD_LIMIT:
            notes.append(
                "OCR text bahut lambi hai; UI me pehla hissa dikh raha hai, poora text TXT download me hai."
            )

        notes = _dedupe(notes)
        result = {
            "id": rid,
            "filename": filename,
            "mode": mode,
            "status": status,
            "pages": len(pages),
            "ocr_chars": len(text),
            "word_count": word_count,
            "confidence": confidence,
            "elapsed": round(time.time() - t0, 2),
            "language": lang_used,
            "changed": bool(changed),
            "text": text[:_TEXT_PAYLOAD_LIMIT],
            "text_truncated": len(text) > _TEXT_PAYLOAD_LIMIT,
            "match_total": len(selected),
            "matches": match_payload,
            "styles": styles[:300],
            "page_previews": previews,
            "preview_truncated": len(pages) > _MAX_PREVIEW_PAGES,
            "downloads": downloads,
            "notes": notes,
        }
        result["_text_full"] = text
        result["_file_ids"] = [
            f["id"] for f in sess["files"] if f["id"] in {d["id"] for d in downloads}
        ]
        sess["results"].append(result)
        combined = _write_combined(sess)
        result["combined"] = combined
        # internal fields are not sent to the browser
        public = {k: v for k, v in result.items() if not k.startswith("_")}
        public["ok"] = True
        return public
    except Exception as exc:
        return {"ok": False, "error": _friendly_error(exc)}


def _cb_download(token, fid):
    try:
        sess = _get_session(token)
        rec = _find_file(sess, fid)
        if not rec:
            raise ValueError("File session me nahi mili. Dobara process karein.")
        path = Path(rec["path"])
        if not path.exists():
            raise ValueError("File delete ho gayi. Dobara process karein.")
        size = path.stat().st_size
        if size > _MAX_DOWNLOAD:
            raise ValueError("File bahut badi hai (55 MB se zyada). Chhota format ya kam resolution use karein.")
        data = path.read_bytes()
        return {
            "ok": True,
            "filename": rec["filename"],
            "mime": rec["mime"],
            "b64": base64.b64encode(data).decode("ascii"),
        }
    except Exception as exc:
        return {"ok": False, "error": _friendly_error(exc)}


def _cb_zip(token):
    try:
        sess = _get_session(token)
        if not sess["results"]:
            raise ValueError("Pehle koi file process karein.")
        zip_path = Path(sess["dir"]) / "textswap_results.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for r in sess["results"]:
                folder = f"{_safe_stem(r['filename'], 'file')}_{r['id']}"
                for fid in r.get("_file_ids", []):
                    rec = _find_file(sess, fid)
                    if not rec:
                        continue
                    comp = (
                        zipfile.ZIP_STORED
                        if rec["mime"].startswith(("image/", "application/pdf"))
                        else zipfile.ZIP_DEFLATED
                    )
                    z.write(rec["path"], f"{folder}/{rec['filename']}", compress_type=comp)
            combined = _find_file(sess, sess.get("combined_id")) if sess.get("combined_id") else None
            if combined:
                z.write(combined["path"], "all_ocr_text.txt")
            lines = [
                "TextSwap Studio - Results",
                f"Files: {len(sess['results'])}",
                f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}",
                "",
                "Har file ka folder alag hai. OCR text, report aur edited image usi folder me hain.",
                "Edit: Google Colab notebook session.",
            ]
            z.writestr("README.txt", "\n".join(lines))
        size = zip_path.stat().st_size
        if size > _MAX_ZIP:
            raise ValueError("ZIP bahut bada hai. Individual files download karein.")
        data = zip_path.read_bytes()
        return {
            "ok": True,
            "filename": f"textswap_results_{time.strftime('%Y%m%d_%H%M')}.zip",
            "mime": "application/zip",
            "b64": base64.b64encode(data).decode("ascii"),
        }
    except Exception as exc:
        return {"ok": False, "error": _friendly_error(exc)}


def _cb_reset(token):
    try:
        _get_session(token)
        sess = _SESSIONS.pop(token, None)
        if sess:
            shutil.rmtree(sess.get("dir", ""), ignore_errors=True)
        _new_session(token)
        return {"ok": True}
    except Exception as exc:
        return {"ok": False, "error": _friendly_error(exc)}


_HTML_TEMPLATE = r"""
<div id="ts-app">
  <style>
    #ts-app, #ts-app * { box-sizing: border-box; }
    #ts-app {
      --ink:#0f172a; --muted:#64748b; --line:#e7ecf4; --brand:#4f46e5; --brand2:#7c3aed;
      --ok:#047857; --warn:#b45309; --danger:#b91c1c;
      font-family: "Segoe UI", Inter, system-ui, -apple-system, BlinkMacSystemFont, Arial, sans-serif;
      color: var(--ink);
      background:
        radial-gradient(1100px 520px at 6% -10%, #e0e7ff 0%, rgba(224,231,255,0) 60%),
        radial-gradient(900px 460px at 100% 0%, #fce7f3 0%, rgba(252,231,243,0) 55%),
        #f6f7fb;
      border-radius: 24px;
      padding: 20px;
      line-height: 1.45;
      position: relative;
    }
    #ts-app h1, #ts-app h2, #ts-app h3, #ts-app p { margin: 0; }
    #ts-app button { font-family: inherit; }
    #ts-app .ts-hero {
      background: linear-gradient(120deg, #0f172a 0%, #312e81 58%, #6d28d9 100%);
      color: #fff; border-radius: 20px; padding: 20px 22px;
      display: flex; align-items: center; gap: 16px; flex-wrap: wrap;
      box-shadow: 0 18px 40px rgba(49,46,129,.22);
      position: relative; overflow: hidden;
    }
    #ts-app .ts-hero::after {
      content:""; position:absolute; right:-60px; top:-80px; width:260px; height:260px;
      background:radial-gradient(circle, rgba(255,255,255,.18), transparent 65%); border-radius:50%;
    }
    #ts-app .ts-brand { display:flex; align-items:center; gap:14px; position:relative; z-index:1; flex:1; min-width:260px; }
    #ts-app .ts-logo {
      width:56px; height:56px; display:grid; place-items:center; font-size:29px;
      background: linear-gradient(135deg, rgba(255,255,255,.24), rgba(255,255,255,.08));
      border:1px solid rgba(255,255,255,.28); border-radius:17px;
      box-shadow: inset 0 1px 0 rgba(255,255,255,.25);
      flex:0 0 auto;
    }
    #ts-app .ts-hero h1 { font-size:25px; line-height:1.1; letter-spacing:-.4px; }
    #ts-app .ts-hero p { color:#c7d2fe; font-size:12.5px; margin-top:4px; }
    #ts-app .ts-badges { display:flex; gap:6px; flex-wrap:wrap; position:relative; z-index:1; }
    #ts-app .ts-badges span {
      font-size:10.5px; font-weight:700; color:#e0e7ff; background:rgba(255,255,255,.12);
      border:1px solid rgba(255,255,255,.18); padding:5px 8px; border-radius:999px;
    }
    #ts-app .ts-style-note {
      margin-top:11px; border:1px solid #c7d2fe; border-radius:13px; padding:10px 11px;
      background:linear-gradient(135deg,#eef2ff,#f5f3ff); color:#3730a3;
      display:flex; gap:9px; align-items:flex-start; font-size:11px; line-height:1.45;
    }
    #ts-app .ts-style-note .ico {
      width:27px; height:27px; border-radius:8px; background:#fff; display:grid;
      place-items:center; font-size:15px; flex:0 0 auto; box-shadow:0 1px 4px rgba(67,56,202,.12);
    }
    #ts-app .ts-style-note b { display:block; font-size:11.5px; }
    #ts-app .ts-style-note span { display:block; color:#4f46e5; font-size:10.6px; margin-top:1px; }
    #ts-app .ts-subhead {
      font-size:10.2px; font-weight:900; color:#64748b; text-transform:uppercase;
      letter-spacing:.7px; margin:2px 0 8px; padding-top:2px;
    }
    #ts-app .ts-subhead:not(:first-child) { margin-top:13px; border-top:1px solid #f1f5f9; padding-top:12px; }
    #ts-app .ts-banner { border-radius:12px; padding:10px 12px; font-size:12px; margin-top:12px; line-height:1.55; }
    #ts-app .ts-banner.err { background:#fef2f2; border:1px solid #fecaca; color:#991b1b; }
    #ts-app .ts-banner.info { background:#eff6ff; border:1px solid #bfdbfe; color:#1e40af; }
    #ts-app .ts-banner.warn { background:#fffbeb; border:1px solid #fde68a; color:#92400e; }
    #ts-app .ts-grid { display:grid; grid-template-columns: minmax(340px, 430px) minmax(0, 1fr); gap:16px; margin-top:16px; align-items:start; }
    #ts-app .ts-panel {
      background:#fff; border:1px solid #e9edf5; border-radius:19px;
      box-shadow: 0 12px 28px rgba(15,23,42,.055);
    }
    #ts-app .ts-controls { padding:15px; }
    #ts-app .ts-results { padding:15px; min-height:420px; }
    #ts-app .ts-modes { display:grid; grid-template-columns:1fr 1fr; background:#eef2ff; padding:4px; border-radius:14px; gap:4px; }
    #ts-app .ts-modes button {
      border:0; background:transparent; padding:10px 8px; border-radius:11px;
      font-weight:800; font-size:12.5px; color:#64748b; cursor:pointer; transition:.18s;
    }
    #ts-app .ts-modes button.active { background:#fff; color:#4338ca; box-shadow:0 3px 10px rgba(67,56,202,.12); }
    #ts-app .ts-drop {
      margin-top:12px; border:2px dashed #c7d2fe; border-radius:16px; padding:18px 14px;
      text-align:center; background:linear-gradient(180deg,#fbfdff,#f5f8ff);
      cursor:pointer; transition:.18s; outline:none;
    }
    #ts-app .ts-drop:hover, #ts-app .ts-drop.drag { border-color:#6366f1; background:#eef2ff; }
    #ts-app .ts-drop-ico { font-size:30px; line-height:1; margin-bottom:6px; }
    #ts-app .ts-drop-title { font-size:13.5px; font-weight:800; color:#3730a3; }
    #ts-app .ts-drop .ts-hint { margin-top:4px; }
    #ts-app .ts-hint { font-size:10.8px; color:#94a3b8; line-height:1.55; }
    #ts-app .ts-files { display:flex; flex-direction:column; gap:7px; margin-top:9px; }
    #ts-app .ts-file {
      display:flex; align-items:center; gap:9px; padding:7px; border:1px solid #eef2f7;
      border-radius:12px; background:#fff; min-width:0;
    }
    #ts-app .ts-file img, #ts-app .ts-file .ph {
      width:42px; height:42px; border-radius:8px; object-fit:cover; background:#eef2ff;
      flex:0 0 auto; display:grid; place-items:center; font-size:19px;
    }
    #ts-app .ts-file .fm { min-width:0; flex:1; }
    #ts-app .ts-file .fn { font-size:11.5px; font-weight:700; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
    #ts-app .ts-file .fs { font-size:10px; color:#94a3b8; margin-top:1px; }
    #ts-app .ts-x {
      border:0; background:#fef2f2; color:#b91c1c; width:26px; height:26px; border-radius:8px;
      cursor:pointer; font-weight:800; flex:0 0 auto; font-size:11px;
    }
    #ts-app .ts-sec { border:1px solid #edf1f7; border-radius:14px; margin-top:11px; overflow:hidden; background:#fff; }
    #ts-app .ts-sec > summary {
      list-style:none; cursor:pointer; padding:11px 12px; display:flex; align-items:center; gap:7px;
      font-weight:800; font-size:12.2px; color:#1e293b; background:#fbfcff; user-select:none;
    }
    #ts-app .ts-sec > summary::-webkit-details-marker { display:none; }
    #ts-app .ts-sec > summary::after { content:"⌄"; margin-left:auto; color:#94a3b8; font-size:15px; transition:.18s; }
    #ts-app .ts-sec:not([open]) > summary::after { transform:rotate(-90deg); }
    #ts-app .ts-sec-body { padding:11px 12px 12px; border-top:1px solid #f1f5f9; }
    #ts-app .ts-field { margin-bottom:10px; min-width:0; }
    #ts-app .ts-field:last-child { margin-bottom:0; }
    #ts-app .ts-label { display:flex; justify-content:space-between; gap:8px; font-size:11px; font-weight:800; color:#475569; margin-bottom:5px; }
    #ts-app select, #ts-app input[type="text"], #ts-app input[type="number"] {
      width:100%; border:1px solid #dbe3ee; border-radius:10px; padding:8px 9px;
      font-size:12px; background:#fff; color:#0f172a; outline:none; font-family:inherit;
    }
    #ts-app select:focus, #ts-app input[type="text"]:focus, #ts-app input[type="number"]:focus {
      border-color:#818cf8; box-shadow:0 0 0 3px rgba(99,102,241,.12);
    }
    #ts-app input[type="range"] { width:100%; accent-color:#4f46e5; height:18px; }
    #ts-app .ts-row { display:grid; grid-template-columns:1fr 1fr; gap:9px; }
    #ts-app .ts-toggle { display:flex; align-items:center; gap:8px; padding:5px 0; font-size:11.4px; color:#334155; cursor:pointer; min-width:0; }
    #ts-app .ts-toggle span { overflow:hidden; text-overflow:ellipsis; }
    #ts-app .ts-toggle input {
      appearance:none; -webkit-appearance:none; width:33px; height:19px; background:#cbd5e1;
      border-radius:999px; position:relative; cursor:pointer; transition:.18s; flex:0 0 auto; margin:0;
    }
    #ts-app .ts-toggle input:checked { background:#4f46e5; }
    #ts-app .ts-toggle input::after {
      content:""; position:absolute; top:2px; left:2px; width:15px; height:15px;
      border-radius:50%; background:#fff; transition:.18s; box-shadow:0 1px 3px rgba(0,0,0,.22);
    }
    #ts-app .ts-toggle input:checked::after { transform:translateX(14px); }
    #ts-app .ts-langs { display:grid; grid-template-columns:repeat(3, minmax(0,1fr)); gap:5px; max-height:148px; overflow:auto; padding:1px 2px 2px 0; }
    #ts-app .ts-chip {
      display:flex; align-items:center; gap:4px; font-size:10.2px; padding:5px 6px;
      border:1px solid #e2e8f0; border-radius:999px; cursor:pointer; background:#fff;
      white-space:nowrap; overflow:hidden; color:#475569;
    }
    #ts-app .ts-chip.on { background:#eef2ff; border-color:#a5b4fc; color:#3730a3; font-weight:800; }
    #ts-app .ts-chip input { display:none; }
    #ts-app .ts-pair { display:grid; grid-template-columns:minmax(0,1fr) minmax(0,1fr) 26px; gap:5px; align-items:center; margin-bottom:6px; }
    #ts-app .ts-pair input { font-size:11.2px; padding:7px 8px; }
    #ts-app .ts-btn {
      border:0; border-radius:10px; padding:9px 11px; font-weight:800; font-size:12px;
      cursor:pointer; transition:.16s; display:inline-flex; align-items:center; justify-content:center; gap:5px;
    }
    #ts-app .ts-btn.primary {
      background:linear-gradient(135deg,#4f46e5,#7c3aed); color:#fff; width:100%;
      padding:12px; box-shadow:0 8px 20px rgba(79,70,229,.28); font-size:13px;
    }
    #ts-app .ts-btn.primary:hover { filter:brightness(1.06); }
    #ts-app .ts-btn.ghost { background:#f1f5f9; color:#475569; }
    #ts-app .ts-btn.small { background:#eef2ff; color:#4338ca; padding:6px 8px; font-size:10.5px; border-radius:8px; }
    #ts-app .ts-btn:disabled { opacity:.5; cursor:not-allowed; box-shadow:none; }
    #ts-app .ts-action { display:grid; grid-template-columns:1fr auto; gap:7px; margin-top:12px; align-items:stretch; }
    #ts-app .ts-status { margin-top:8px; font-size:11px; color:#475569; min-height:16px; line-height:1.45; }
    #ts-app .ts-progress { height:7px; background:#eef2f7; border-radius:999px; overflow:hidden; margin-top:7px; display:none; }
    #ts-app .ts-progress .bar { height:100%; width:0; background:linear-gradient(90deg,#4f46e5,#a855f7); transition:width .25s; }
    #ts-app .ts-results-head { display:flex; align-items:center; gap:8px; flex-wrap:wrap; padding:1px 1px 8px; }
    #ts-app .ts-results-head h2 { font-size:15px; letter-spacing:-.2px; }
    #ts-app .ts-summary { display:flex; gap:5px; flex-wrap:wrap; }
    #ts-app .ts-stat { background:#f1f5f9; color:#475569; font-size:10px; padding:3px 7px; border-radius:999px; font-weight:800; }
    #ts-app .ts-stat.ok { background:#ecfdf5; color:#047857; }
    #ts-app .ts-stat.bad { background:#fef2f2; color:#b91c1c; }
    #ts-app .ts-dl { margin-left:auto; display:flex; gap:5px; flex-wrap:wrap; }
    #ts-app .ts-empty {
      border:1px dashed #dbe3ee; border-radius:15px; padding:38px 16px; text-align:center;
      color:#94a3b8; font-size:12px; background:#fbfcff;
    }
    #ts-app .ts-empty .big { font-size:36px; margin-bottom:7px; }
    #ts-app .ts-result { border:1px solid #e8ecf3; border-radius:15px; padding:11px; margin-top:9px; background:#fff; }
    #ts-app .ts-result-head { display:flex; gap:8px; align-items:center; min-width:0; }
    #ts-app .ts-result-head .ico {
      width:34px; height:34px; border-radius:9px; background:#eef2ff; display:grid; place-items:center;
      font-size:17px; flex:0 0 auto;
    }
    #ts-app .ts-result-head .ttl { min-width:0; flex:1; }
    #ts-app .ts-result-head h3 { font-size:12px; margin:0; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; }
    #ts-app .ts-meta { font-size:9.8px; color:#94a3b8; margin-top:2px; }
    #ts-app .ts-styles {
      display:flex; gap:5px; flex-wrap:wrap; margin-top:7px;
    }
    #ts-app .ts-styles span {
      font-size:9.4px; font-weight:800; color:#4338ca; background:#eef2ff;
      border:1px solid #c7d2fe; border-radius:999px; padding:3px 7px;
    }
    #ts-app .ts-badge { font-size:9.4px; font-weight:900; padding:4px 7px; border-radius:999px; white-space:nowrap; }
    #ts-app .ts-badge.done { background:#ecfdf5; color:#047857; }
    #ts-app .ts-badge.warn { background:#fffbeb; color:#b45309; }
    #ts-app .ts-badge.ocr { background:#eff6ff; color:#1d4ed8; }
    #ts-app .ts-badge.err { background:#fef2f2; color:#b91c1c; }
    #ts-app .ts-previews { display:grid; grid-template-columns:1fr 1fr; gap:7px; margin-top:9px; }
    #ts-app .ts-previews.single { grid-template-columns:1fr; }
    #ts-app .ts-fig { margin:0; border:1px solid #eef2f7; border-radius:11px; overflow:hidden; background:#f8fafc; min-width:0; }
    #ts-app .ts-fig img { width:100%; display:block; cursor:zoom-in; max-height:290px; object-fit:contain; background:#fff; }
    #ts-app .ts-fig figcaption { font-size:9.4px; color:#64748b; padding:4px 6px; background:#fbfcff; border-top:1px solid #eef2f7; }
    #ts-app .ts-pages { display:flex; gap:4px; flex-wrap:wrap; margin-top:7px; }
    #ts-app .ts-pages button {
      border:1px solid #e2e8f0; background:#fff; border-radius:6px; padding:2px 6px;
      font-size:9px; cursor:pointer; color:#475569;
    }
    #ts-app .ts-pages button.active { background:#4f46e5; color:#fff; border-color:#4f46e5; }
    #ts-app .ts-text summary, #ts-app .ts-matches summary {
      cursor:pointer; font-size:11px; font-weight:800; color:#334155; padding:7px 0;
    }
    #ts-app .ts-text pre {
      max-height:210px; overflow:auto; white-space:pre-wrap; word-break:break-word;
      font-size:10.8px; line-height:1.55; background:#f8fafc; border:1px solid #eef2f7;
      padding:8px; border-radius:8px; margin:4px 0 6px;
      font-family: ui-monospace, SFMono-Regular, Consolas, "Liberation Mono", monospace;
    }
    #ts-app .ts-matches ul { list-style:none; margin:4px 0 0; padding:0; display:flex; flex-direction:column; gap:5px; }
    #ts-app .ts-matches li { font-size:10.2px; background:#fbfcff; border:1px solid #eef2f7; border-radius:8px; padding:6px 7px; }
    #ts-app .ts-matches mark { background:#fef08a; padding:0 1px; border-radius:2px; }
    #ts-app .ts-matches .ctx { color:#64748b; margin-top:2px; line-height:1.5; word-break:break-word; }
    #ts-app .ts-matches .pairtag { display:inline-block; background:#eef2ff; color:#4338ca; font-size:8.6px; font-weight:900; border-radius:999px; padding:2px 5px; margin-right:4px; }
    #ts-app .ts-downloads { display:flex; flex-wrap:wrap; gap:5px; margin-top:7px; }
    #ts-app .ts-downloads .ts-btn { background:#f8fafc; border:1px solid #e2e8f0; color:#334155; font-size:9.6px; padding:5px 7px; }
    #ts-app .ts-notes { margin-top:8px; background:#fffbeb; border:1px solid #fde68a; color:#92400e; border-radius:9px; padding:7px 8px; font-size:10.2px; line-height:1.55; }
    #ts-app .ts-foot { text-align:center; color:#94a3b8; font-size:10px; margin-top:14px; line-height:1.6; }
    #ts-app .ts-single-note { font-size:10px; color:#94a3b8; margin-top:5px; }
    #ts-app code { background:#f1f5f9; padding:1px 4px; border-radius:4px; font-size:.95em; }
    #ts-toast { position:fixed; right:18px; bottom:18px; z-index:99999; display:flex; flex-direction:column; gap:7px; pointer-events:none; }
    #ts-toast .ts-toast {
      background:#111827; color:#fff; padding:9px 12px; border-radius:10px; font-size:11.5px;
      box-shadow:0 10px 30px rgba(0,0,0,.25); opacity:0; transform:translateY(7px);
      transition:.2s; max-width:340px; font-family:"Segoe UI",system-ui,sans-serif;
    }
    #ts-toast .ts-toast.show { opacity:1; transform:none; }
    #ts-toast .ts-toast.ok { background:#065f46; }
    #ts-toast .ts-toast.err { background:#991b1b; }
    #ts-toast .ts-toast.info { background:#1e3a8a; }
    #ts-modal {
      position:fixed; inset:0; background:rgba(15,23,42,.74); z-index:99998; display:none;
      align-items:center; justify-content:center; padding:16px; font-family:"Segoe UI",system-ui,sans-serif;
    }
    #ts-modal.show { display:flex; }
    #ts-modal .ts-modal-box {
      background:#fff; border-radius:15px; max-width:1100px; max-height:92vh; overflow:auto;
      padding:10px; position:relative; min-width:280px;
    }
    #ts-modal .ts-modal-box img { max-width:100%; max-height:82vh; display:block; margin:auto; border-radius:8px; }
    #ts-modal .ts-modal-box pre { white-space:pre-wrap; word-break:break-word; font-size:11.5px; max-height:82vh; overflow:auto; margin:0; }
    #ts-modal .ts-modal-close {
      position:sticky; top:0; float:right; border:0; background:#f1f5f9; color:#334155;
      width:30px; height:30px; border-radius:8px; cursor:pointer; font-weight:900; z-index:1;
    }
    @media (max-width: 980px) {
      #ts-app .ts-grid { grid-template-columns: 1fr; }
      #ts-app .ts-results { min-height: 260px; }
    }
    @media (max-width: 620px) {
      #ts-app { padding:10px; border-radius:0; }
      #ts-app .ts-hero { padding:16px; }
      #ts-app .ts-hero h1 { font-size:21px; }
      #ts-app .ts-previews { grid-template-columns: 1fr; }
      #ts-app .ts-langs { grid-template-columns: repeat(2, minmax(0,1fr)); }
      #ts-app .ts-dl { margin-left:0; width:100%; }
    }
  </style>

  <div class="ts-hero">
    <div class="ts-brand">
      <div class="ts-logo">🪄</div>
      <div>
        <h1>TextSwap Studio</h1>
        <p>Text replace karein — font, size aur colour bilkul source jaisa.</p>
      </div>
    </div>
    <div class="ts-badges">
      <span>✨ Auto style match</span><span>Batch + PDF</span>
    </div>
  </div>

  <div id="ts-runtime"></div>

  <div class="ts-grid">
    <section class="ts-panel ts-controls">
      <div class="ts-modes" role="tablist">
        <button type="button" class="active" data-mode="swap">✏️ Find &amp; Replace</button>
        <button type="button" data-mode="ocr">🔍 Sirf OCR</button>
      </div>

      <div class="ts-drop" id="ts-drop" tabindex="0" role="button" aria-label="Upload images or PDF">
        <input type="file" id="ts-file" multiple accept="image/*,.pdf,application/pdf" style="display:none">
        <div class="ts-drop-ico">📤</div>
        <div class="ts-drop-title" id="ts-droptitle">Images / PDF yahan drop karein</div>
        <div class="ts-hint">ya click karke select karein · multiple files · PDF supported · max <span id="ts-maxmb">30</span> MB/file</div>
      </div>
      <div class="ts-files" id="ts-files"></div>

      <div class="ts-style-note">
        <div class="ico">✨</div>
        <div>
          <b>Source ka style automatic copy hota hai</b>
          <span>Font family, bold/italic, size, baseline, colour, letter width, case aur scan softness original text se match kiye jaate hain.</span>
        </div>
      </div>

      <details class="ts-sec" id="ts-pairs-sec" open>
        <summary>✏️ Kya replace karna hai? <span id="ts-paircount" class="ts-badge ocr">0 pair</span></summary>
        <div class="ts-sec-body">
          <div id="ts-pairs"></div>
          <button type="button" class="ts-btn small" id="ts-addpair">＋ Pair add karein</button>
          <div class="ts-hint" style="margin-top:7px">Find text OCR se match hota hai. Replace khali chhodne se text erase ho jayega.</div>
        </div>
      </details>

      <details class="ts-sec" id="ts-output-sec">
        <summary>💾 Output</summary>
        <div class="ts-sec-body">
          <div id="ts-outputwrap">
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Format</span></div>
                <select id="ts-format">
                  <option value="png">PNG (lossless)</option>
                  <option value="jpg">JPG (small)</option>
                  <option value="webp">WEBP</option>
                  <option value="pdf">PDF</option>
                </select>
              </div>
              <div class="ts-field">
                <div class="ts-label"><span>Quality</span><span id="ts-qualv">92</span></div>
                <input type="range" id="ts-quality" min="60" max="100" step="1" value="92">
              </div>
            </div>
            <div class="ts-field">
              <div class="ts-label"><span>Filename prefix</span></div>
              <input type="text" id="ts-prefix" placeholder="edited_{name}" maxlength="80">
              <div class="ts-hint">Khali = original filename. <code>{name}</code> original name se replace hota hai.</div>
            </div>
          </div>
          <label class="ts-toggle"><input type="checkbox" id="ts-boxes" checked><span>Preview me detected boxes dikhayein</span></label>
          <div id="ts-ocrnote" class="ts-banner info" style="display:none; margin-top:8px">OCR mode: annotated PNG + TXT/JSON report milega. Replacement settings ignore honge.</div>
        </div>
      </details>

      <details class="ts-sec" id="ts-advanced">
        <summary>⚙️ Advanced settings</summary>
        <div class="ts-sec-body">
          <div class="ts-subhead">OCR</div>
          <div class="ts-field">
            <div class="ts-label"><span>Preset</span></div>
            <select id="ts-preset">
              <option value="photo">📷 Photo / scan (recommended)</option>
              <option value="clean">📄 Clean document</option>
              <option value="sparse">🧾 Sparse text</option>
              <option value="id">🪪 ID / certificate</option>
              <option value="hindi">🇮🇳 Hindi + English</option>
              <option value="custom">⚙️ Custom</option>
            </select>
          </div>
          <div class="ts-field">
            <div class="ts-label"><span>Language</span><span class="ts-hint" id="ts-lang-count">1 selected</span></div>
            <div class="ts-langs" id="ts-langs"></div>
            <div class="ts-hint" style="margin-top:5px">Nayi language pehli baar 10–30 sec le sakti hai (auto install).</div>
          </div>
          <div class="ts-row">
            <div class="ts-field">
              <div class="ts-label"><span>PSM mode</span></div>
              <select id="ts-psm">
                <option value="3">3 — Auto (default)</option>
                <option value="4">4 — Single column</option>
                <option value="6">6 — Single block</option>
                <option value="11">11 — Sparse text</option>
                <option value="12">12 — Sparse + OSD</option>
                <option value="13">13 — Raw line</option>
                <option value="8">8 — Single word</option>
                <option value="10">10 — Single char</option>
              </select>
            </div>
            <div class="ts-field">
              <div class="ts-label"><span>PDF DPI</span></div>
              <select id="ts-dpi">
                <option value="150">150 — fast</option>
                <option value="200">200</option>
                <option value="300" selected>300 — default</option>
                <option value="400">400 — sharp</option>
              </select>
            </div>
          </div>
          <label class="ts-toggle"><input type="checkbox" id="ts-enhance" checked><span>Auto-enhance OCR</span></label>
          <div class="ts-row">
            <label class="ts-toggle"><input type="checkbox" id="ts-contrast" checked><span>Auto contrast</span></label>
            <label class="ts-toggle"><input type="checkbox" id="ts-denoise"><span>Denoise (slow)</span></label>
          </div>
          <div class="ts-row">
            <label class="ts-toggle"><input type="checkbox" id="ts-gray"><span>Grayscale</span></label>
            <label class="ts-toggle"><input type="checkbox" id="ts-bin"><span>Binarize</span></label>
          </div>
          <div class="ts-row">
            <div class="ts-field">
              <div class="ts-label"><span>Binarize mode</span></div>
              <select id="ts-binmode"><option value="otsu">Otsu</option><option value="adaptive">Adaptive</option></select>
            </div>
            <label class="ts-toggle"><input type="checkbox" id="ts-invert"><span>Invert colors</span></label>
          </div>
          <div class="ts-row">
            <div class="ts-field">
              <div class="ts-label"><span>Min confidence</span><span id="ts-confv">0</span></div>
              <input type="range" id="ts-conf" min="0" max="95" step="5" value="0">
            </div>
            <div class="ts-field">
              <div class="ts-label"><span>PDF max pages</span></div>
              <input type="number" id="ts-maxpages" min="0" max="50" value="0">
            </div>
          </div>

          <div id="ts-swaponly">
            <div class="ts-subhead">Style match</div>
            <div class="ts-field">
              <div class="ts-label"><span>Font</span><span class="ts-hint">Auto recommended</span></div>
              <select id="ts-font"><option value="">✨ Auto detect (best)</option></select>
              <div class="ts-hint">Auto har match ka font, bold/italic aur size source se copy karta hai.</div>
            </div>
            <div class="ts-row">
              <label class="ts-toggle"><input type="checkbox" id="ts-preserve" checked><span>Exact source size</span></label>
              <label class="ts-toggle"><input type="checkbox" id="ts-weight" checked><span>Match text weight</span></label>
            </div>
            <div class="ts-row">
              <label class="ts-toggle"><input type="checkbox" id="ts-stretch" checked><span>Match letter width</span></label>
              <label class="ts-toggle"><input type="checkbox" id="ts-skew" checked><span>Follow text tilt</span></label>
            </div>
            <div class="ts-field">
              <label class="ts-toggle"><input type="checkbox" id="ts-case-match" checked><span>Source case match</span></label>
              <div class="ts-hint" style="margin-top:-2px">Source ALL CAPS / Title Case replacement par copy hota hai.</div>
            </div>
            <div class="ts-field">
              <label class="ts-toggle"><input type="checkbox" id="ts-pixel-lock" checked><span>🔬 Exact pixel lock</span></label>
              <div class="ts-hint" style="margin-top:-2px">Har character ki height source ke barabar lock hoti hai — 200 px source kabhi 201 px nahi banega. Find text pehle characters me split hota hai, phir draw hota hai.</div>
            </div>
            <div class="ts-subhead">Watermark / logo / stamp</div>
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Layer</span></div>
                <select id="ts-wm-mode">
                  <option value="auto">🛡️ Auto detect (best)</option>
                  <option value="under">Under text</option>
                  <option value="over">Over text</option>
                  <option value="off">Off</option>
                </select>
              </div>
              <div class="ts-field">
                <div class="ts-label"><span>Sensitivity</span><span id="ts-wmv">0.5</span></div>
                <input type="range" id="ts-wm-sens" min="0" max="1" step="0.05" value="0.5">
              </div>
            </div>
            <div class="ts-hint" style="margin-top:-3px">Text ke peeche watermark/logo ho to Auto rakhein: sirf old glyph strokes erase honge, visible watermark pixels protect rahenge.</div>
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Text color</span></div>
                <div style="display:flex;gap:6px;align-items:center">
                  <label class="ts-toggle" style="padding:0"><input type="checkbox" id="ts-colorauto" checked><span>Auto</span></label>
                  <input type="color" id="ts-color" value="#111111" disabled style="width:38px;height:30px;padding:1px;border:1px solid #dbe3ee;border-radius:7px;background:#fff">
                </div>
              </div>
              <label class="ts-toggle"><input type="checkbox" id="ts-softauto" checked><span>Auto edge softness</span></label>
            </div>
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Manual size</span><span id="ts-scalev">1.00×</span></div>
                <input type="range" id="ts-scale" min="0.5" max="2" step="0.05" value="1">
              </div>
              <div class="ts-field">
                <div class="ts-label"><span>Manual softness</span><span id="ts-softv">0.5</span></div>
                <input type="range" id="ts-soft" min="0" max="2" step="0.1" value="0.5" disabled>
              </div>
            </div>
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Position nudge X</span><span class="ts-hint">px</span></div>
                <input type="number" id="ts-nudgex" min="-20" max="20" step="0.5" value="0">
              </div>
              <div class="ts-field">
                <div class="ts-label"><span>Position nudge Y</span><span class="ts-hint">px</span></div>
                <input type="number" id="ts-nudgey" min="-20" max="20" step="0.5" value="0">
              </div>
            </div>
            <div class="ts-hint" style="margin-top:-3px">Auto alignment perfect na lage to X/Y se halka adjust karein.</div>
            <div class="ts-row">
              <div class="ts-field">
                <div class="ts-label"><span>Erase method</span></div>
                <select id="ts-method">
                  <option value="auto">Auto (best)</option>
                  <option value="fill">Fill (paper match)</option>
                  <option value="inpaint">Inpaint</option>
                </select>
              </div>
              <div class="ts-field">
                <div class="ts-label"><span>Text align</span></div>
                <select id="ts-align"><option value="left">Left</option><option value="center">Center</option><option value="right">Right</option></select>
              </div>
            </div>
            <div class="ts-row">
              <label class="ts-toggle"><input type="checkbox" id="ts-shrink"><span>Gap me fit karein</span></label>
              <label class="ts-toggle"><input type="checkbox" id="ts-first"><span>Sirf pehla match</span></label>
            </div>
            <div class="ts-row">
              <label class="ts-toggle"><input type="checkbox" id="ts-case"><span>Case sensitive</span></label>
              <label class="ts-toggle"><input type="checkbox" id="ts-whole"><span>Whole word</span></label>
            </div>
            <div class="ts-field">
              <div class="ts-label"><span>Fuzzy match (OCR galtiyan)</span><span id="ts-fuzzyv">0.80</span></div>
              <input type="range" id="ts-fuzzy" min="0.5" max="1" step="0.01" value="0.8">
              <div class="ts-hint">1.0 = exact only. Kam value = zyada tolerant, par galat match ka risk.</div>
            </div>
          </div>
        </div>
      </details>

      <div class="ts-action">
        <button type="button" class="ts-btn primary" id="ts-run">🚀 Process karein</button>
        <button type="button" class="ts-btn ghost" id="ts-reset">Reset</button>
      </div>
      <div class="ts-status" id="ts-status"></div>
      <div class="ts-progress" id="ts-progress"><div class="bar" id="ts-bar"></div></div>
    </section>

    <section class="ts-panel ts-results">
      <div class="ts-results-head">
        <h2>📊 Results</h2>
        <div class="ts-summary" id="ts-summary"></div>
        <div class="ts-dl">
          <button type="button" class="ts-btn small" id="ts-combined" style="display:none">📄 Combined text</button>
          <button type="button" class="ts-btn small" id="ts-zip" disabled>⬇️ Sab ZIP</button>
        </div>
      </div>
      <div class="ts-empty" id="ts-empty">
        <div class="big">🖼️</div>
        <div><b>Abhi koi result nahi hai.</b></div>
        <div style="margin-top:5px">Images upload karein, find &amp; replace likhein, aur <b>Process</b> dabayein.</div>
      </div>
      <div id="ts-results"></div>
    </section>
  </div>

  <div class="ts-foot">🔒 Files sirf is Colab session ke temp folder me rehti hain; reset / reload par delete.</div>
</div>

<div id="ts-toast"></div>
<div id="ts-modal" aria-hidden="true">
  <div class="ts-modal-box">
    <button type="button" class="ts-modal-close" id="ts-modal-close">✕</button>
    <img id="ts-modal-img" alt="preview" style="display:none">
    <pre id="ts-modal-text" style="display:none"></pre>
  </div>
</div>

<script>
window.TS_CONFIG = __CONFIG__;
</script>
<script>
(function () {
  "use strict";
  const CFG = window.TS_CONFIG || {};
  const $ = (id) => document.getElementById(id);
  const state = {
    mode: "swap",
    files: [],
    results: [],
    errors: [],
    busy: false,
    seq: 0,
    combined: null
  };
  const DLMAP = new Map();
  const MAXFILES = CFG.maxFiles || 12;
  const MAXBYTES = (CFG.maxMB || 30) * 1024 * 1024;
  const PRESETS = {
    photo: { langs: ["eng"], psm: 3, enhance: true, pre: { auto_contrast: true, denoise: true, grayscale: false, binarize: false, invert: false } },
    clean: { langs: ["eng"], psm: 3, enhance: false, pre: { auto_contrast: false, denoise: false, grayscale: false, binarize: false, invert: false } },
    sparse: { langs: ["eng"], psm: 11, enhance: true, pre: { auto_contrast: true, denoise: false, grayscale: false, binarize: false, invert: false } },
    id: { langs: ["eng"], psm: 6, enhance: true, pre: { auto_contrast: true, denoise: false, grayscale: true, binarize: false, invert: false } },
    hindi: { langs: ["eng", "hin"], psm: 3, enhance: true, pre: { auto_contrast: true, denoise: true, grayscale: false, binarize: false, invert: false } }
  };

  function esc(s) {
    return String(s == null ? "" : s).replace(/[&<>"']/g, function (c) {
      return { "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c];
    });
  }
  function fmtBytes(n) {
    n = Number(n) || 0;
    if (n < 1024) return n + " B";
    if (n < 1048576) return (n / 1024).toFixed(1) + " KB";
    return (n / 1048576).toFixed(1) + " MB";
  }
  function fmtNum(n) {
    n = Number(n) || 0;
    return n.toLocaleString("en-IN");
  }
  function ctxHtml(s) {
    return esc(s).replace(/\[\[/g, "<mark>").replace(/\]\]/g, "</mark>");
  }
  function toast(msg, type, ms) {
    const wrap = $("ts-toast");
    if (!wrap) return;
    const el = document.createElement("div");
    el.className = "ts-toast " + (type || "info");
    el.textContent = String(msg);
    wrap.appendChild(el);
    requestAnimationFrame(function () { el.classList.add("show"); });
    setTimeout(function () {
      el.classList.remove("show");
      setTimeout(function () { el.remove(); }, 220);
    }, ms || 3800);
  }
  let fitTimer = null;
  let lastH = 0;
  function fit() {
    clearTimeout(fitTimer);
    fitTimer = setTimeout(function () {
      try {
        if (window.google && google.colab && google.colab.output && google.colab.output.setIframeHeight) {
          const h = Math.max(720, document.documentElement.scrollHeight + 14);
          if (Math.abs(h - lastH) > 8) {
            lastH = h;
            google.colab.output.setIframeHeight(h, true);
          }
        }
      } catch (e) { /* ignore */ }
    }, 120);
  }
  function remember(downloads) {
    (downloads || []).forEach(function (d) { if (d && d.id) DLMAP.set(d.id, d); });
  }
  function saveBlob(blob, filename) {
    const a = document.createElement("a");
    const url = URL.createObjectURL(blob);
    a.href = url;
    a.download = filename || "download";
    document.body.appendChild(a);
    a.click();
    a.remove();
    setTimeout(function () { URL.revokeObjectURL(url); }, 12000);
  }
  function saveB64(b64, mime, filename) {
    const bin = atob(b64);
    const len = bin.length;
    const arr = new Uint8Array(len);
    for (let i = 0; i < len; i++) arr[i] = bin.charCodeAt(i);
    saveBlob(new Blob([arr], { type: mime || "application/octet-stream" }), filename);
  }
  async function callPy(name, args) {
    if (!(window.google && google.colab && google.colab.kernel && google.colab.kernel.invokeFunction)) {
      throw new Error("Colab kernel connection nahi mila. Notebook cell dobara run karein.");
    }
    const res = await google.colab.kernel.invokeFunction(name, args, {});
    const data = res && res.data ? res.data : {};
    let out = data["application/json"];
    if (out === undefined) out = data["text/plain"];
    if (typeof out === "string") {
      let s = out;
      let parsed = false;
      for (let i = 0; i < 2; i++) {
        try {
          const v = JSON.parse(s);
          if (typeof v === "string") { s = v; continue; }
          out = v; parsed = true; break;
        } catch (e) {
          if (i === 0) {
            try { out = JSON.parse(s.replace(/'/g, '"')); parsed = true; } catch (e2) { /* ignore */ }
            if (parsed) break;
          }
        }
      }
      if (!parsed && typeof out === "string") throw new Error("Kernel response parse nahi hua.");
    }
    if (!out || typeof out !== "object") throw new Error("Kernel response invalid hai.");
    return out;
  }
  async function downloadFile(fid) {
    if (!fid) return;
    toast("File taiyaar ho rahi hai…", "info", 1800);
    try {
      const r = await callPy("notebook.textswap_v2_download", [CFG.token, fid]);
      if (!r || !r.ok) throw new Error((r && r.error) || "Download fail");
      const meta = DLMAP.get(fid);
      saveB64(r.b64, r.mime || (meta && meta.mime), r.filename || (meta && meta.filename) || "download");
    } catch (e) {
      toast("Download fail: " + (e.message || e), "err", 5200);
    }
  }
  function fileToB64(file) {
    return new Promise(function (resolve, reject) {
      const reader = new FileReader();
      reader.onload = function () {
        const s = String(reader.result || "");
        const i = s.indexOf(",");
        resolve(i >= 0 ? s.slice(i + 1) : s);
      };
      reader.onerror = function () { reject(reader.error || new Error("File read fail")); };
      reader.readAsDataURL(file);
    });
  }
  function setStatus(msg) { $("ts-status").textContent = msg || ""; fit(); }
  function setProgress(pct, msg) {
    const wrap = $("ts-progress");
    wrap.style.display = pct > 0 || msg ? "block" : "none";
    $("ts-bar").style.width = Math.max(0, Math.min(100, pct)) + "%";
    if (msg) setStatus(msg);
  }
  function setBusy(on) {
    state.busy = !!on;
    $("ts-run").disabled = !!on || !!CFG.runtimeError || !CFG.colab;
    $("ts-reset").disabled = !!on;
    document.querySelectorAll(".ts-modes button").forEach(function (b) { b.disabled = !!on; });
  }
  function runtimeBanner() {
    const box = $("ts-runtime");
    if (CFG.runtimeError) {
      box.innerHTML = '<div class="ts-banner err"><b>⚠️ Engine ready nahi hai.</b><br>' + esc(CFG.runtimeError) + '<br>Notebook cell dobara run karein.</div>';
      return;
    }
    if (!CFG.colab) {
      box.innerHTML = '<div class="ts-banner err"><b>⚠️ Colab kernel connection nahi mila.</b><br>' + esc(CFG.colabError || "Yeh UI Google Colab ke andar chalayein.") + '</div>';
      return;
    }
    if (CFG.runtimeNotes && CFG.runtimeNotes.length) {
      box.innerHTML = '<div class="ts-banner info">ℹ️ ' + CFG.runtimeNotes.map(esc).join("<br>") + '</div>';
    }
  }

  // ---------- languages ----------
  const langWrap = $("ts-langs");
  (CFG.languages || []).forEach(function (l) {
    const lab = document.createElement("label");
    lab.className = "ts-chip" + (l.code === "eng" ? " on" : "");
    lab.innerHTML = '<input type="checkbox" value="' + esc(l.code) + '"' + (l.code === "eng" ? " checked" : "") + '><span>' + esc((l.flag || "") + " " + l.label) + "</span>";
    langWrap.appendChild(lab);
  });
  function updateLangCount() {
    const n = langWrap.querySelectorAll('input[type="checkbox"]:checked').length;
    $("ts-lang-count").textContent = n + " selected";
  }
  langWrap.addEventListener("change", function (e) {
    const cb = e.target.closest('input[type="checkbox"]');
    if (!cb) return;
    cb.closest(".ts-chip").classList.toggle("on", cb.checked);
    updateLangCount();
    if ($("ts-preset").value !== "custom") $("ts-preset").value = "custom";
  });
  function getLangs() {
    return Array.from(langWrap.querySelectorAll('input[type="checkbox"]:checked')).map(function (x) { return x.value; });
  }
  function setLangs(codes) {
    const set = new Set(codes || []);
    langWrap.querySelectorAll('input[type="checkbox"]').forEach(function (cb) {
      cb.checked = set.has(cb.value);
      cb.closest(".ts-chip").classList.toggle("on", cb.checked);
    });
    updateLangCount();
  }

  // ---------- optional font override ----------
  const fontSelect = $("ts-font");
  (CFG.fonts || []).forEach(function (f) {
    const opt = document.createElement("option");
    opt.value = f.path;
    opt.textContent = f.label;
    fontSelect.appendChild(opt);
  });

  // ---------- presets ----------
  $("ts-preset").addEventListener("change", function () {
    const p = PRESETS[this.value];
    if (!p) return;
    setLangs(p.langs);
    $("ts-psm").value = String(p.psm);
    $("ts-enhance").checked = p.enhance;
    $("ts-contrast").checked = p.pre.auto_contrast;
    $("ts-denoise").checked = p.pre.denoise;
    $("ts-gray").checked = p.pre.grayscale;
    $("ts-bin").checked = p.pre.binarize;
    $("ts-invert").checked = p.pre.invert;
    toast("Preset apply ho gaya", "ok", 1600);
  });

  // ---------- pairs ----------
  const pairsWrap = $("ts-pairs");
  function updatePairCount() {
    const n = pairsWrap.children.length;
    $("ts-paircount").textContent = n + " pair" + (n === 1 ? "" : "s");
    $("ts-addpair").disabled = n >= 10;
  }
  function addPair(find, repl) {
    if (pairsWrap.children.length >= 10) { toast("Max 10 pairs", "err"); return; }
    const row = document.createElement("div");
    row.className = "ts-pair";
    row.innerHTML = '<input type="text" class="find" placeholder="Find: Rehan Khan" maxlength="400"><input type="text" class="repl" placeholder="Replace: Amanullah (khali = erase)" maxlength="400"><button type="button" class="ts-x" title="Remove">✕</button>';
    row.querySelector(".find").value = find || "";
    row.querySelector(".repl").value = repl || "";
    row.querySelector(".ts-x").addEventListener("click", function () {
      row.remove();
      updatePairCount();
      if (!pairsWrap.children.length) addPair();
    });
    pairsWrap.appendChild(row);
    updatePairCount();
  }
  $("ts-addpair").addEventListener("click", function () { addPair(); });
  function getPairs() {
    return Array.from(pairsWrap.querySelectorAll(".ts-pair")).map(function (row) {
      return {
        find: row.querySelector(".find").value.trim(),
        replace: row.querySelector(".repl").value
      };
    });
  }

  // ---------- files ----------
  const fileInput = $("ts-file");
  const drop = $("ts-drop");
  const filesWrap = $("ts-files");
  drop.addEventListener("click", function (e) {
    if (e.target === fileInput || e.target.closest(".ts-file")) return;
    fileInput.click();
  });
  fileInput.addEventListener("click", function (e) { e.stopPropagation(); });
  drop.addEventListener("keydown", function (e) {
    if (e.key === "Enter" || e.key === " ") { e.preventDefault(); fileInput.click(); }
  });
  fileInput.addEventListener("change", function () { addFiles(fileInput.files); fileInput.value = ""; });
  ["dragenter", "dragover"].forEach(function (ev) {
    drop.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); drop.classList.add("drag"); });
  });
  ["dragleave", "drop"].forEach(function (ev) {
    drop.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); drop.classList.remove("drag"); });
  });
  drop.addEventListener("drop", function (e) {
    if (e.dataTransfer && e.dataTransfer.files) addFiles(e.dataTransfer.files);
  });
  function addFiles(list) {
    const arr = Array.from(list || []);
    for (const f of arr) {
      if (state.files.length >= MAXFILES) { toast("Max " + MAXFILES + " files ek batch me.", "err"); break; }
      if (f.size > MAXBYTES) { toast(f.name + " " + (CFG.maxMB || 30) + " MB se badi hai.", "err"); continue; }
      const ext = (f.name.split(".").pop() || "").toLowerCase();
      const isPdf = f.type === "application/pdf" || ext === "pdf";
      const isImg = f.type.indexOf("image/") === 0 || ["png", "jpg", "jpeg", "webp", "bmp", "tif", "tiff", "gif", "jfif"].indexOf(ext) >= 0;
      if (!isPdf && !isImg) { toast(f.name + " supported format nahi hai.", "err"); continue; }
      if (state.files.some(function (x) { return x.file.name === f.name && x.file.size === f.size && x.file.lastModified === f.lastModified; })) continue;
      state.files.push({ key: "f" + (++state.seq), file: f, url: isPdf ? null : URL.createObjectURL(f) });
    }
    renderFiles();
  }
  function removeFile(key) {
    const idx = state.files.findIndex(function (x) { return x.key === key; });
    if (idx < 0) return;
    const item = state.files[idx];
    if (item.url) URL.revokeObjectURL(item.url);
    state.files.splice(idx, 1);
    renderFiles();
  }
  function renderFiles() {
    filesWrap.innerHTML = "";
    state.files.forEach(function (item) {
      const card = document.createElement("div");
      card.className = "ts-file";
      const thumb = item.url ? '<img src="' + item.url + '" alt="">' : '<div class="ph">📄</div>';
      card.innerHTML = thumb + '<div class="fm"><div class="fn">' + esc(item.file.name) + '</div><div class="fs">' + fmtBytes(item.file.size) + (item.file.type === "application/pdf" ? " · PDF" : "") + '</div></div><button type="button" class="ts-x" title="Remove">✕</button>';
      card.querySelector(".ts-x").addEventListener("click", function (e) { e.stopPropagation(); removeFile(item.key); });
      filesWrap.appendChild(card);
    });
    $("ts-droptitle").textContent = state.files.length
      ? state.files.length + " file" + (state.files.length === 1 ? "" : "s") + " ready — aur add kar sakte hain"
      : "Images / PDF yahan drop karein";
    fit();
  }

  // ---------- mode / controls ----------
  function setMode(mode) {
    state.mode = mode === "ocr" ? "ocr" : "swap";
    document.querySelectorAll(".ts-modes button").forEach(function (b) {
      b.classList.toggle("active", b.getAttribute("data-mode") === state.mode);
    });
    $("ts-pairs-sec").style.display = state.mode === "swap" ? "" : "none";
    $("ts-swaponly").style.display = state.mode === "swap" ? "" : "none";
    $("ts-outputwrap").style.display = state.mode === "swap" ? "" : "none";
    $("ts-ocrnote").style.display = state.mode === "ocr" ? "" : "none";
    document.querySelector(".ts-style-note").style.display = state.mode === "swap" ? "" : "none";
    $("ts-run").textContent = state.mode === "swap" ? "🚀 Process karein" : "🔍 OCR nikalein";
    fit();
  }
  document.querySelectorAll(".ts-modes button").forEach(function (b) {
    b.addEventListener("click", function () { if (!state.busy) setMode(b.getAttribute("data-mode")); });
  });

  function syncQuality() {
    const f = $("ts-format").value;
    $("ts-quality").disabled = (f === "png" || f === "pdf");
  }
  $("ts-format").addEventListener("change", syncQuality);
  $("ts-quality").addEventListener("input", function () { $("ts-qualv").textContent = this.value; });
  $("ts-conf").addEventListener("input", function () { $("ts-confv").textContent = this.value; });
  $("ts-scale").addEventListener("input", function () { $("ts-scalev").textContent = Number(this.value).toFixed(2) + "×"; });
  $("ts-soft").addEventListener("input", function () { $("ts-softv").textContent = Number(this.value).toFixed(1); });
  $("ts-fuzzy").addEventListener("input", function () { $("ts-fuzzyv").textContent = Number(this.value).toFixed(2); });
  $("ts-wm-sens").addEventListener("input", function () { $("ts-wmv").textContent = Number(this.value).toFixed(2); });
  $("ts-colorauto").addEventListener("change", function () { $("ts-color").disabled = this.checked; });
  function syncSoft() { $("ts-soft").disabled = $("ts-softauto").checked; }
  $("ts-softauto").addEventListener("change", syncSoft);
  $("ts-preserve").addEventListener("change", function () { if (this.checked) $("ts-shrink").checked = false; });
  $("ts-shrink").addEventListener("change", function () { if (this.checked) $("ts-preserve").checked = false; });
  syncSoft();
  syncQuality();
  ["ts-psm", "ts-enhance", "ts-contrast", "ts-denoise", "ts-gray", "ts-bin", "ts-binmode", "ts-invert"].forEach(function (id) {
    $(id).addEventListener("change", function () { if ($("ts-preset").value !== "custom") $("ts-preset").value = "custom"; });
  });

  function collectSettings(langs, pairs) {
    return {
      mode: state.mode,
      langs: langs,
      psm: parseInt($("ts-psm").value, 10),
      enhance: $("ts-enhance").checked,
      preprocess: {
        auto_contrast: $("ts-contrast").checked,
        denoise: $("ts-denoise").checked,
        grayscale: $("ts-gray").checked,
        binarize: $("ts-bin").checked,
        binarize_mode: $("ts-binmode").value,
        invert: $("ts-invert").checked
      },
      pairs: state.mode === "swap" ? pairs : [],
      fuzzy: parseFloat($("ts-fuzzy").value),
      case_sensitive: $("ts-case").checked,
      whole_word: $("ts-whole").checked,
      first_only: $("ts-first").checked,
      font_override: $("ts-font").value || null,
      method: $("ts-method").value,
      align: $("ts-align").value,
      shrink: $("ts-shrink").checked,
      preserve_size: $("ts-preserve").checked,
      case_match: $("ts-case-match").checked,
      pixel_lock: $("ts-pixel-lock").checked,
      watermark_mode: $("ts-wm-mode").value,
      watermark_sensitivity: parseFloat($("ts-wm-sens").value),
      weight_match: $("ts-weight").checked,
      stretch_match: $("ts-stretch").checked,
      nudge_x: parseFloat($("ts-nudgex").value) || 0,
      nudge_y: parseFloat($("ts-nudgey").value) || 0,
      soften: parseFloat($("ts-soft").value),
      soften_auto: $("ts-softauto").checked,
      auto_skew: $("ts-skew").checked,
      color: $("ts-colorauto").checked ? null : $("ts-color").value,
      size_scale: parseFloat($("ts-scale").value),
      output_format: $("ts-format").value,
      quality: parseInt($("ts-quality").value, 10),
      name_prefix: $("ts-prefix").value.trim(),
      dpi: parseInt($("ts-dpi").value, 10),
      max_pages: Math.max(0, parseInt($("ts-maxpages").value, 10) || 0),
      box_preview: $("ts-boxes").checked,
      min_conf: parseInt($("ts-conf").value, 10)
    };
  }

  // ---------- results ----------
  function resultMeta(r) {
    const bits = [];
    bits.push((r.pages || 1) + " page" + ((r.pages || 1) === 1 ? "" : "s"));
    if (r.ocr_chars) bits.push(fmtNum(r.ocr_chars) + " chars");
    if (r.confidence) bits.push(r.confidence + "% conf");
    if (r.elapsed != null) bits.push(r.elapsed + "s");
    if (r.language) bits.push("OCR: " + r.language);
    return bits.join(" · ");
  }
  function badgeFor(r) {
    if (r.status === "done") return '<span class="ts-badge done">✅ Ho gaya</span>';
    if (r.status === "no_match") return '<span class="ts-badge warn">⚠️ Match nahi mila</span>';
    if (r.status === "no_change") return '<span class="ts-badge warn">⚠️ Change nahi hua</span>';
    if (r.status === "no_text") return '<span class="ts-badge warn">⚠️ Text nahi mila</span>';
    return '<span class="ts-badge ocr">🔍 OCR ready</span>';
  }
  function previewFigures(r) {
    const pv = (r.page_previews && r.page_previews[0]) || { before: "", after: "" };
    const showAfter = r.mode === "ocr" || (r.mode === "swap" && r.status === "done" && r.changed);
    const beforeLabel = r.mode === "ocr" ? "OCR overlay" : "Pehle (detected boxes)";
    const afterLabel = r.mode === "ocr" ? "Original" : "Baad me";
    let html = '<div class="ts-previews' + (showAfter ? "" : " single") + '" data-rid="' + esc(r.id) + '">';
    html += '<figure class="ts-fig"><img data-zoom="1" data-kind="before" data-rid="' + esc(r.id) + '" src="' + esc(pv.before) + '" alt="before"><figcaption>' + esc(beforeLabel) + '</figcaption></figure>';
    if (showAfter) {
      html += '<figure class="ts-fig"><img data-zoom="1" data-kind="after" data-rid="' + esc(r.id) + '" src="' + esc(pv.after) + '" alt="after"><figcaption>' + esc(afterLabel) + '</figcaption></figure>';
    }
    html += '</div>';
    return html;
  }
  function pageTabs(r) {
    if (!r.page_previews || r.page_previews.length <= 1) return "";
    let html = '<div class="ts-pages" data-rid="' + esc(r.id) + '">';
    r.page_previews.forEach(function (p, i) {
      html += '<button type="button" data-page="' + i + '" data-rid="' + esc(r.id) + '" class="' + (i === 0 ? "active" : "") + '">Page ' + (p.page || (i + 1)) + '</button>';
    });
    html += "</div>";
    return html;
  }
  function downloadsHtml(r) {
    if (!r.downloads || !r.downloads.length) return "";
    return '<div class="ts-downloads">' + r.downloads.map(function (d) {
      return '<button type="button" class="ts-btn" data-dl="' + esc(d.id) + '">⬇️ ' + esc(d.label) + ' <span style="opacity:.6">' + fmtBytes(d.size) + '</span></button>';
    }).join("") + "</div>";
  }
  function matchesHtml(r) {
    if (!r.matches || !r.matches.length) return "";
    const list = r.matches.map(function (m, i) {
      const score = m.score >= 0.999 ? "exact" : Math.round(m.score * 100) + "%";
      return '<li><span class="pairtag">#' + (m.pair || 1) + '</span><b>' + esc(m.matched_text) + '</b> <span style="color:#94a3b8">· page ' + (m.page || 1) + ' · ' + esc(score) + '</span><div class="ctx">' + ctxHtml(m.context || "") + "</div></li>";
    }).join("");
    return '<details class="ts-matches"><summary>🎯 Matches (' + (r.match_total || r.matches.length) + ')</summary><ul>' + list + "</ul></details>";
  }
  function notesHtml(r) {
    if (!r.notes || !r.notes.length) return "";
    return '<div class="ts-notes">ℹ️ ' + r.notes.map(esc).join("<br>") + "</div>";
  }
  function stylesHtml(r) {
    const list = (r.styles || []).filter(function (s) { return s && s.font; });
    if (!list.length) return "";
    const seen = new Set();
    const chips = [];
    list.forEach(function (s) {
      const key = s.font + "|" + s.size;
      if (seen.has(key)) return;
      seen.add(key);
      const title = "style match " + Math.round((s.score || 0) * 100) + "% · " + (s.color || "") + (s.weight ? " · weight " + s.weight : "") + (s.case ? " · case " + s.case : "");
      chips.push('<span title="' + esc(title) + '">✨ ' + esc(s.font) + " · " + esc(s.size) + " px</span>");
    });
    const wm = list.find(function (s) { return s.watermark && s.watermark.present; });
    if (wm) {
      chips.push('<span title="underlay-aware erase">🛡️ Watermark ' + esc(wm.watermark.order) + " · " + Math.round((wm.watermark.confidence || 0) * 100) + "%</span>");
    }
    const px = list.find(function (s) { return s.pixel; });
    if (px && Number.isFinite(px.pixel.source_reconstruction_iou)) {
      const p = px.pixel;
      const src = p.source_core_bbox || [];
      const dst = p.replacement_ink_bbox_derotated || p.replacement_ink_bbox || [];
      const dims = function (b) { return b.length === 4 ? (b[2] - b[0]) + "×" + (b[3] - b[1]) + " px" : "unknown"; };
      const title = "source core " + dims(src) + " · replacement ink " + dims(dst) + " · baseline offset " + esc(p.baseline_offset) + " px · scale " + esc(p.replacement_scale_x) + "×" + esc(p.replacement_scale_y) + " · tracking " + esc(p.tracking) + " px";
      chips.push('<span title="' + esc(title) + '">🔬 source IoU ' + Math.round(p.source_reconstruction_iou * 100) + "% · whole-line</span>");
    }
    return '<div class="ts-styles">' + chips.slice(0, 6).join("") + "</div>";
  }
  function resultCard(r) {
    const text = r.text || "";
    const preview = text.length > 3000 ? text.slice(0, 3000) + "\n…" : text;
    return '<article class="ts-result" data-rid="' + esc(r.id) + '">' +
      '<div class="ts-result-head"><div class="ico">' + (r.mode === "ocr" ? "🔍" : "✏️") + '</div>' +
      '<div class="ttl"><h3 title="' + esc(r.filename) + '">' + esc(r.filename) + '</h3><div class="ts-meta">' + esc(resultMeta(r)) + '</div></div>' +
      badgeFor(r) + '</div>' +
      '<div class="ts-single-note">' + (r.match_total ? "🎯 " + r.match_total + " match" + (r.match_total === 1 ? "" : "es") + " mile" : "") + '</div>' +
      stylesHtml(r) + previewFigures(r) + pageTabs(r) +
      '<details class="ts-text"><summary>📝 OCR Text (' + fmtNum((r.text || "").length) + ' chars)</summary><pre>' + esc(preview) + '</pre>' +
      '<div style="display:flex;gap:5px;flex-wrap:wrap"><button type="button" class="ts-btn small" data-copy="' + esc(r.id) + '">📋 Copy</button>' +
      '<button type="button" class="ts-btn small" data-full="' + esc(r.id) + '">🔎 Full text</button></div></details>' +
      matchesHtml(r) + notesHtml(r) + downloadsHtml(r) + '</article>';
  }
  function errorCard(e) {
    return '<article class="ts-result"><div class="ts-result-head"><div class="ico" style="background:#fef2f2">⚠️</div><div class="ttl"><h3>' + esc(e.filename) + '</h3><div class="ts-meta">Process fail</div></div><span class="ts-badge err">Error</span></div><div class="ts-notes" style="background:#fef2f2;border-color:#fecaca;color:#991b1b">' + esc(e.error) + '</div></article>';
  }
  function renderSummary() {
    const box = $("ts-summary");
    if (!state.results.length && !state.errors.length) { box.innerHTML = ""; return; }
    let chars = 0, matches = 0, secs = 0;
    state.results.forEach(function (r) { chars += r.ocr_chars || 0; matches += r.match_total || 0; secs += r.elapsed || 0; });
    let html = '<span class="ts-stat">' + state.results.length + " file" + (state.results.length === 1 ? "" : "s") + "</span>";
    if (matches) html += '<span class="ts-stat ok">🎯 ' + matches + " match" + (matches === 1 ? "" : "es") + "</span>";
    if (chars) html += '<span class="ts-stat">' + fmtNum(chars) + " chars</span>";
    if (secs) html += '<span class="ts-stat">' + secs.toFixed(1) + "s</span>";
    if (state.errors.length) html += '<span class="ts-stat bad">' + state.errors.length + " fail</span>";
    box.innerHTML = html;
  }
  function renderAll() {
    const empty = $("ts-empty");
    const wrap = $("ts-results");
    const has = state.results.length || state.errors.length;
    empty.style.display = has ? "none" : "";
    wrap.innerHTML = state.results.map(resultCard).join("") + state.errors.map(errorCard).join("");
    $("ts-zip").disabled = !state.results.length;
    $("ts-combined").style.display = state.combined ? "" : "none";
    renderSummary();
    fit();
  }

  $("ts-results").addEventListener("click", function (e) {
    const dl = e.target.closest("[data-dl]");
    if (dl) { downloadFile(dl.getAttribute("data-dl")); return; }
    const zoom = e.target.closest("img[data-zoom]");
    if (zoom) { openModal({ img: zoom.src, title: zoom.getAttribute("alt") || "Preview" }); return; }
    const copy = e.target.closest("[data-copy]");
    if (copy) {
      const r = state.results.find(function (x) { return x.id === copy.getAttribute("data-copy"); });
      if (r) copyText(r.text || "");
      return;
    }
    const full = e.target.closest("[data-full]");
    if (full) {
      const r = state.results.find(function (x) { return x.id === full.getAttribute("data-full"); });
      if (r) openModal({ text: r.text || "", title: r.filename });
      return;
    }
    const page = e.target.closest("[data-page]");
    if (page) {
      const rid = page.getAttribute("data-rid");
      const idx = parseInt(page.getAttribute("data-page"), 10) || 0;
      const r = state.results.find(function (x) { return x.id === rid; });
      if (!r || !r.page_previews || !r.page_previews[idx]) return;
      const pv = r.page_previews[idx];
      const card = page.closest(".ts-result");
      card.querySelectorAll("img[data-kind='before']").forEach(function (img) { img.src = pv.before; });
      card.querySelectorAll("img[data-kind='after']").forEach(function (img) { img.src = pv.after; });
      page.closest(".ts-pages").querySelectorAll("button").forEach(function (b) { b.classList.toggle("active", b === page); });
      return;
    }
  });

  function copyText(text) {
    if (navigator.clipboard && navigator.clipboard.writeText) {
      navigator.clipboard.writeText(text).then(function () { toast("Text copy ho gaya", "ok", 1800); }, function () { fallbackCopy(text); });
    } else {
      fallbackCopy(text);
    }
  }
  function fallbackCopy(text) {
    const ta = document.createElement("textarea");
    ta.value = text;
    ta.style.position = "fixed";
    ta.style.opacity = "0";
    document.body.appendChild(ta);
    ta.select();
    try { document.execCommand("copy"); toast("Text copy ho gaya", "ok", 1800); }
    catch (e) { toast("Copy fail; manual select karein", "err"); }
    ta.remove();
  }

  // ---------- modal ----------
  function openModal(opts) {
    const modal = $("ts-modal");
    const img = $("ts-modal-img");
    const pre = $("ts-modal-text");
    if (opts.img) { img.src = opts.img; img.style.display = "block"; pre.style.display = "none"; pre.textContent = ""; }
    else { pre.textContent = opts.text || ""; pre.style.display = "block"; img.style.display = "none"; img.src = ""; }
    modal.classList.add("show");
    modal.setAttribute("aria-hidden", "false");
  }
  function closeModal() {
    $("ts-modal").classList.remove("show");
    $("ts-modal").setAttribute("aria-hidden", "true");
    $("ts-modal-img").src = "";
  }
  $("ts-modal-close").addEventListener("click", closeModal);
  $("ts-modal").addEventListener("click", function (e) { if (e.target === this) closeModal(); });
  document.addEventListener("keydown", function (e) { if (e.key === "Escape") closeModal(); });

  // ---------- run / reset ----------
  async function run() {
    if (state.busy) return;
    if (!CFG.colab) { toast("Colab kernel connection nahi hai. Cell dobara run karein.", "err", 5000); return; }
    if (CFG.runtimeError) { toast("Engine ready nahi hai. Cell dobara run karein.", "err", 5000); return; }
    if (!state.files.length) { toast("Pehle image/PDF upload karein.", "err"); return; }
    const langs = getLangs();
    if (!langs.length) { toast("Kam se kam ek OCR language chunein.", "err"); return; }
    let pairs = [];
    if (state.mode === "swap") {
      pairs = getPairs().filter(function (p) { return p.find; });
      if (!pairs.length) { toast("Kam se kam ek 'Find' text likhein (ya Sirf OCR mode chunein).", "err", 4500); return; }
    }
    const settings = collectSettings(langs, pairs);
    state.busy = true;
    setBusy(true);
    setProgress(0, "Session reset ho raha hai…");
    try {
      const rr = await callPy("notebook.textswap_v2_reset", [CFG.token]);
      if (!rr || !rr.ok) throw new Error((rr && rr.error) || "Reset fail");
    } catch (e) {
      // A reset failure should not block a fresh run; the backend will create a session if needed.
      console.warn(e);
    }
    state.results = [];
    state.errors = [];
    state.combined = null;
    DLMAP.clear();
    renderAll();
    const total = state.files.length;
    for (let i = 0; i < total; i++) {
      const item = state.files[i];
      setProgress((i / total) * 100, (i + 1) + "/" + total + " · " + item.file.name + " — OCR chal raha hai…");
      try {
        const b64 = await fileToB64(item.file);
        const res = await callPy("notebook.textswap_v2_process", [CFG.token, JSON.stringify({ filename: item.file.name, b64: b64, settings: settings })]);
        if (!res || !res.ok) throw new Error((res && res.error) || "Unknown error");
        state.results.push(res);
        if (res.combined) { state.combined = res.combined; remember([res.combined]); }
        remember(res.downloads || []);
        renderAll();
      } catch (e) {
        state.errors.push({ filename: item.file.name, error: String(e && e.message ? e.message : e) });
        renderAll();
      }
      setProgress(((i + 1) / total) * 100, (i + 1) + "/" + total + " complete");
    }
    state.busy = false;
    setBusy(false);
    const matches = state.results.reduce(function (a, r) { return a + (r.match_total || 0); }, 0);
    if (state.results.length) {
      setStatus("✅ " + state.results.length + " file(s) ho gayi · " + matches + " match(es)" + (state.errors.length ? " · " + state.errors.length + " fail" : ""));
      toast("Processing complete", "ok");
    } else {
      setStatus("❌ Koi file process nahi hui. Error details results me dekhein.");
      toast("Processing fail hui", "err", 5000);
    }
    renderAll();
  }
  $("ts-run").addEventListener("click", run);

  async function resetAll() {
    if (state.busy) return;
    try { await callPy("notebook.textswap_v2_reset", [CFG.token]); } catch (e) { /* ignore */ }
    state.files.forEach(function (f) { if (f.url) URL.revokeObjectURL(f.url); });
    state.files = [];
    state.results = [];
    state.errors = [];
    state.combined = null;
    DLMAP.clear();
    pairsWrap.innerHTML = "";
    addPair();
    setMode("swap");
    $("ts-advanced").open = false;
    $("ts-output-sec").open = false;
    renderFiles();
    renderAll();
    setStatus("");
    setProgress(0, "");
    toast("Reset ho gaya", "ok", 1800);
  }
  $("ts-reset").addEventListener("click", resetAll);

  $("ts-zip").addEventListener("click", async function () {
    if (!state.results.length) return;
    toast("ZIP ban rahi hai…", "info", 2200);
    try {
      const r = await callPy("notebook.textswap_v2_zip", [CFG.token]);
      if (!r || !r.ok) throw new Error((r && r.error) || "ZIP fail");
      saveB64(r.b64, r.mime || "application/zip", r.filename || "textswap_results.zip");
    } catch (e) { toast("ZIP fail: " + (e.message || e), "err", 5200); }
  });
  $("ts-combined").addEventListener("click", function () { if (state.combined) downloadFile(state.combined.id); });

  // ---------- init ----------
  runtimeBanner();
  addPair();
  syncQuality();
  setMode("swap");
  renderFiles();
  renderAll();
  updateLangCount();
  setBusy(false);
  if (window.ResizeObserver) {
    new ResizeObserver(function () { fit(); }).observe($("ts-app"));
  }
  document.querySelectorAll("details").forEach(function (d) { d.addEventListener("toggle", fit); });
  setTimeout(fit, 250);
  setTimeout(fit, 900);
})();
</script>
"""


# =============================================================================
# TextSwap Studio - Colab app wiring
# =============================================================================
import json as _json

try:
    from google.colab import output as _colab_output

    _COLAB = True
    _COLAB_ERROR = ""
except Exception as _colab_exc:
    _colab_output = None
    _COLAB = False
    _COLAB_ERROR = f"Google Colab kernel connection nahi mila ({_colab_exc}). Notebook ko Colab me kholein."


def _json_response(payload):
    try:
        from IPython.display import JSON

        return JSON(payload)
    except Exception:
        return payload


def _cb_process_display(token, payload):
    return _json_response(_cb_process(token, payload))


def _cb_download_display(token, fid):
    return _json_response(_cb_download(token, fid))


def _cb_zip_display(token):
    return _json_response(_cb_zip(token))


def _cb_reset_display(token):
    return _json_response(_cb_reset(token))


_APP_TOKEN = uuid.uuid4().hex
_APP_INIT_ERROR = ""
try:
    _new_session(_APP_TOKEN)
except Exception as _app_exc:
    _APP_INIT_ERROR = f"Temp session create nahi hua: {_app_exc}"

if _COLAB:
    try:
        _colab_output.register_callback("notebook.textswap_v2_process", _cb_process_display)
        _colab_output.register_callback("notebook.textswap_v2_download", _cb_download_display)
        _colab_output.register_callback("notebook.textswap_v2_zip", _cb_zip_display)
        _colab_output.register_callback("notebook.textswap_v2_reset", _cb_reset_display)
    except Exception as _reg_exc:
        _COLAB = False
        _COLAB_ERROR = f"UI callback register nahi hua: {_reg_exc}"

_CONFIG = {
    "token": _APP_TOKEN,
    "version": "2.0",
    "colab": bool(_COLAB and not _COLAB_ERROR),
    "colabError": _COLAB_ERROR,
    "runtimeError": str(globals().get("_RUNTIME_ERROR") or _APP_INIT_ERROR or ""),
    "runtimeNotes": list(globals().get("_RUNTIME_NOTES") or [])[:6],
    "maxFiles": _MAX_FILES,
    "maxMB": _MAX_UPLOAD // (1024 * 1024),
    "languages": _LANG_OPTIONS,
    "fonts": _font_choices()[:90],
}

_APP_HTML = _HTML_TEMPLATE.replace(
    "__CONFIG__", _json.dumps(_CONFIG, ensure_ascii=False).replace("</", "<\\/")
)

try:
    from IPython.display import HTML, clear_output, display

    clear_output(wait=True)
    display(HTML(_APP_HTML))
except Exception:
    # Last-resort fallback for a non-IPython kernel.
    print("TextSwap Studio needs Google Colab / IPython. Please open this notebook in Colab.")
